<a href="https://colab.research.google.com/github/19kaokkk/store/blob/main/01_Data_Audit_and_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

# E-commerce Recommendation System
## 01. Data Audit and Cleaning

### Research Topic
Xây dựng hệ thống gợi ý sản phẩm thương mại điện tử
dựa trên hành vi tuần tự và đa hành vi của khách hàng

### Pipeline
- ML1: Customer Segmentation
- ML2: Purchase Intention Prediction
- ML3: Sequential & Multi-Behavior Recommendation
- ML4: Federated Learning
- ML5: Privacy-Preserving Recommendation


In [ ]:
import zipfile
import os

zip_path = "/content/archive.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Đã giải nén thành công!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/archive.zip'

In [ ]:
os.listdir("/content/data")

FileNotFoundError: [Errno 2] No such file or directory: '/content/data'

In [ ]:
for root, dirs, files in os.walk("/content/data"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
events_path = "/content/data/events.csv"

events = pd.read_csv(events_path)

print("Đọc dữ liệu thành công!")
print("Số dòng:", events.shape[0])
print("Số cột:", events.shape[1])

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/events.csv'

In [ ]:
events.head()

NameError: name 'events' is not defined

In [ ]:
events.info()
events.dtypes

NameError: name 'events' is not defined

In [ ]:
print("Shape:", events.shape)
print("\nColumns:")
print(events.columns.tolist())

NameError: name 'events' is not defined

In [ ]:
missing = events.isnull().sum()

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing / len(events) * 100
})

missing_df

In [ ]:
event_counts = events["event"].value_counts()

event_counts

In [ ]:
event_percent = (
    events["event"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

event_percent

In [ ]:
print("Unique users:", events["visitorid"].nunique())
print("Unique items:", events["itemid"].nunique())
print("Unique transactions:", events["transactionid"].nunique())

In [ ]:
duplicate_count = events.duplicated().sum()

print("Số dòng trùng hoàn toàn:", duplicate_count)

In [ ]:
print("Timestamp nhỏ nhất:", events["timestamp"].min())
print("Timestamp lớn nhất:", events["timestamp"].max())

NameError: name 'events' is not defined

In [ ]:
events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms"
)

print("Thời gian bắt đầu:", events["datetime"].min())
print("Thời gian kết thúc:", events["datetime"].max())

In [ ]:
# ============================================
# BƯỚC 17 — TIẾP TỤC SAU KHI SỬA LỖI
# ============================================

# 17.4. Kiểm tra transactionid theo từng loại event

transaction_check = events.groupby("event").agg(
    total_rows=("transactionid", "size"),
    missing_transactionid=("transactionid", lambda x: x.isna().sum()),
    unique_transactionid=("transactionid", "nunique")
)

print("Transaction ID theo từng loại event:")
display(transaction_check)


# --------------------------------------------
# 17.5. Transaction event nhưng thiếu transactionid
# --------------------------------------------

transaction_missing = events[
    (events["event"] == "transaction") &
    (events["transactionid"].isna())
]

print(
    "\nSố transaction event nhưng transactionid bị thiếu:",
    len(transaction_missing)
)

display(transaction_missing.head(10))


# --------------------------------------------
# 17.6. View/Addtocart nhưng lại có transactionid
# --------------------------------------------

unexpected_transactionid = events[
    (events["event"].isin(["view", "addtocart"])) &
    (events["transactionid"].notna())
]

print(
    "\nSố view/addtocart nhưng lại có transactionid:",
    len(unexpected_transactionid)
)

display(unexpected_transactionid.head(10))


# --------------------------------------------
# 17.7. Kiểm tra itemid
# --------------------------------------------

print("\n========== KIỂM TRA ITEMID ==========")

print("itemid nhỏ nhất:", events["itemid"].min())
print("itemid lớn nhất:", events["itemid"].max())
print("itemid <= 0:", (events["itemid"] <= 0).sum())


# --------------------------------------------
# 17.8. Kiểm tra visitorid
# --------------------------------------------

print("\n========== KIỂM TRA VISITORID ==========")

print("visitorid nhỏ nhất:", events["visitorid"].min())
print("visitorid lớn nhất:", events["visitorid"].max())
print("visitorid <= 0:", (events["visitorid"] <= 0).sum())


# --------------------------------------------
# 17.9. Kiểm tra timestamp
# --------------------------------------------

print("\n========== KIỂM TRA TIMESTAMP ==========")

print("timestamp nhỏ nhất:", events["timestamp"].min())
print("timestamp lớn nhất:", events["timestamp"].max())
print("timestamp <= 0:", (events["timestamp"] <= 0).sum())


# --------------------------------------------
# 17.10. Kiểm tra datetime
# --------------------------------------------

# Nếu datetime chưa tồn tại thì tạo
if "datetime" not in events.columns:
    events["datetime"] = pd.to_datetime(
        events["timestamp"],
        unit="ms"
    )

print("\n========== KIỂM TRA DATETIME ==========")

print("Datetime min:", events["datetime"].min())
print("Datetime max:", events["datetime"].max())


# --------------------------------------------
# 17.11. Kiểm tra transactionid <= 0
# --------------------------------------------

invalid_transactionid = events[
    events["transactionid"].notna() &
    (events["transactionid"] <= 0)
]

print(
    "\nSố transactionid <= 0:",
    len(invalid_transactionid)
)

display(invalid_transactionid.head(10))


# --------------------------------------------
# 17.12. TỔNG HỢP
# --------------------------------------------

print("\n" + "=" * 60)
print("TỔNG HỢP KIỂM TRA BƯỚC 17")
print("=" * 60)

print("1. Duplicate hoàn toàn:", duplicate_count)
print("2. Transaction thiếu transactionid:", len(transaction_missing))
print("3. View/Addtocart có transactionid:", len(unexpected_transactionid))
print("4. itemid <= 0:", (events["itemid"] <= 0).sum())
print("5. visitorid <= 0:", (events["visitorid"] <= 0).sum())
print("6. timestamp <= 0:", (events["timestamp"] <= 0).sum())
print("7. transactionid <= 0:", len(invalid_transactionid))

NameError: name 'events' is not defined

In [ ]:
# ============================================
# BƯỚC 18 — KIỂM TRA VISITORID = 0
# ============================================

visitor_zero = events[events["visitorid"] == 0].copy()

print("Số dòng visitorid = 0:", len(visitor_zero))

display(visitor_zero)
# Kiểm tra 3 dòng visitorid = 0 thuộc event nào

print("\nPhân bố event:")
print(visitor_zero["event"].value_counts())

print("\nThông tin transactionid:")
display(
    visitor_zero[
        ["timestamp", "datetime", "visitorid", "event", "itemid", "transactionid"]
    ]
)
# Kiểm tra visitorid = 0 có bị duplicate hay không

print(
    "Số dòng visitorid = 0 bị duplicate:",
    visitor_zero.duplicated().sum()
)

In [ ]:
# ============================================
# BƯỚC 19 — DATA CLEANING CHÍNH THỨC
# ============================================

# Tạo bản sao dữ liệu gốc
events_clean = events.copy()

print("Số dòng ban đầu:", len(events_clean))


# --------------------------------------------
# 19.1. Xóa duplicate hoàn toàn
# --------------------------------------------

before_duplicate = len(events_clean)

events_clean = events_clean.drop_duplicates()

after_duplicate = len(events_clean)

print("\nDuplicate được xóa:",
      before_duplicate - after_duplicate)


# --------------------------------------------
# 19.2. Xóa visitorid = 0
# --------------------------------------------

before_visitor = len(events_clean)

events_clean = events_clean[
    events_clean["visitorid"] > 0
]

after_visitor = len(events_clean)

print("visitorid = 0 được xóa:",
      before_visitor - after_visitor)


# --------------------------------------------
# 19.3. Xóa transactionid <= 0
# --------------------------------------------

before_transaction = len(events_clean)

events_clean = events_clean[
    ~(
        events_clean["transactionid"].notna() &
        (events_clean["transactionid"] <= 0)
    )
]

after_transaction = len(events_clean)

print("transactionid <= 0 được xóa:",
      before_transaction - after_transaction)


# --------------------------------------------
# 19.4. Reset index
# --------------------------------------------

events_clean = events_clean.reset_index(drop=True)


# --------------------------------------------
# 19.5. Kiểm tra kết quả
# --------------------------------------------

print("\n" + "=" * 60)
print("KẾT QUẢ SAU DATA CLEANING")
print("=" * 60)

print("Số dòng ban đầu:", len(events))
print("Số dòng sau cleaning:", len(events_clean))
print("Tổng số dòng đã xóa:", len(events) - len(events_clean))

print("\nSố duplicate còn lại:",
      events_clean.duplicated().sum())

print("Số visitorid <= 0 còn lại:",
      (events_clean["visitorid"] <= 0).sum())

print(
    "Số transactionid <= 0 còn lại:",
    (
        events_clean["transactionid"].notna() &
        (events_clean["transactionid"] <= 0)
    ).sum()
)

NameError: name 'events' is not defined

In [ ]:
# ============================================
# BƯỚC 20 — VALIDATION SAU CLEANING
# ============================================

print("========== SHAPE ==========")
print(events_clean.shape)


print("\n========== EVENT ==========")
display(events_clean["event"].value_counts())


print("\n========== EVENT % ==========")
display(
    events_clean["event"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


print("\n========== UNIQUE ==========")

print(
    "Unique users:",
    events_clean["visitorid"].nunique()
)

print(
    "Unique items:",
    events_clean["itemid"].nunique()
)

print(
    "Unique transactions:",
    events_clean["transactionid"].nunique()
)


print("\n========== MISSING ==========")

missing_clean = pd.DataFrame({
    "missing_count": events_clean.isna().sum(),
    "missing_percent": (
        events_clean.isna().mean() * 100
    ).round(2)
})

display(missing_clean)


print("\n========== DUPLICATE ==========")

print(
    "Duplicate hoàn toàn:",
    events_clean.duplicated().sum()
)


print("\n========== DATETIME ==========")

print(
    "Min:",
    events_clean["datetime"].min()
)

print(
    "Max:",
    events_clean["datetime"].max()
)

In [ ]:
# ============================================
# BƯỚC 21 — CHUẨN HÓA EVENT + TIMESTAMP
# ============================================

# Chuẩn hóa event
events_clean["event"] = (
    events_clean["event"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Kiểm tra lại event
print("Các giá trị event sau chuẩn hóa:")
print(events_clean["event"].value_counts())

print("\nCác event duy nhất:")
print(events_clean["event"].unique())


# --------------------------------------------
# Kiểm tra timestamp
# --------------------------------------------

print("\nKiểu dữ liệu timestamp:")
print(events_clean["timestamp"].dtype)

print("\nKiểu dữ liệu datetime:")
print(events_clean["datetime"].dtype)


# Kiểm tra datetime có bị NaT không
print(
    "\nSố datetime bị thiếu:",
    events_clean["datetime"].isna().sum()
)

In [ ]:
# ============================================
# BƯỚC 22 — SẮP XẾP THEO USER + TIMESTAMP
# ============================================

events_clean = events_clean.sort_values(
    by=["visitorid", "datetime", "timestamp"]
).reset_index(drop=True)

print("Đã sắp xếp dữ liệu theo visitorid + datetime.")


# Kiểm tra 20 dòng đầu
display(events_clean.head(20))

NameError: name 'events_clean' is not defined

In [ ]:
# ============================================
# BƯỚC 23 — KIỂM TRA THỨ TỰ THỜI GIAN
# ============================================

# Khoảng cách thời gian giữa các event của cùng user
time_diff = (
    events_clean
    .groupby("visitorid")["datetime"]
    .diff()
)

# Kiểm tra có trường hợp thời gian giảm không
negative_diff = (time_diff < pd.Timedelta(0)).sum()

print(
    "Số trường hợp timestamp giảm trong cùng user:",
    negative_diff
)

print(
    "Khoảng thời gian nhỏ nhất giữa 2 event:",
    time_diff.min()
)

print(
    "Khoảng thời gian lớn nhất giữa 2 event:",
    time_diff.max()
)

In [ ]:
# ============================================
# BƯỚC 24 — PHÂN TÍCH KHOẢNG CÁCH EVENT
# ============================================

# Khoảng cách giữa event hiện tại và event trước
time_diff = (
    events_clean
    .groupby("visitorid")["datetime"]
    .diff()
)

# Đổi sang phút
time_diff_minutes = time_diff.dt.total_seconds() / 60

# Loại bỏ giá trị đầu tiên của mỗi user (NaN)
time_diff_minutes = time_diff_minutes.dropna()

print("Số khoảng cách thời gian được phân tích:",
      len(time_diff_minutes))


# --------------------------------------------
# Các percentile quan trọng
# --------------------------------------------

percentiles = time_diff_minutes.quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("\nCác percentile của khoảng cách giữa event:")
display(percentiles)


# --------------------------------------------
# Các ngưỡng phổ biến
# --------------------------------------------

thresholds = [1, 5, 10, 15, 30, 60, 120]

threshold_results = []

for t in thresholds:
    threshold_results.append({
        "threshold_minutes": t,
        "events_within_threshold": (
            time_diff_minutes <= t
        ).sum(),
        "percent_within_threshold": round(
            (time_diff_minutes <= t).mean() * 100,
            2
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print("\nTỷ lệ event nằm trong các ngưỡng thời gian:")
display(threshold_df)


# --------------------------------------------
# Khoảng cách lớn hơn 30 phút
# --------------------------------------------

print(
    "\nSố khoảng cách > 30 phút:",
    (time_diff_minutes > 30).sum()
)

print(
    "Tỷ lệ khoảng cách > 30 phút:",
    round(
        (time_diff_minutes > 30).mean() * 100,
        2
    ),
    "%"
)

In [ ]:
# ============================================
# BƯỚC 25 — SO SÁNH SESSION TIMEOUT
# ============================================

session_test = events_clean[
    ["visitorid", "datetime"]
].copy()

session_test["time_diff"] = (
    session_test
    .groupby("visitorid")["datetime"]
    .diff()
)

session_test["diff_minutes"] = (
    session_test["time_diff"]
    .dt.total_seconds() / 60
)


def count_sessions(timeout_minutes):

    new_session = (
        session_test["diff_minutes"].isna() |
        (session_test["diff_minutes"] > timeout_minutes)
    )

    return new_session.sum()


session_results = []

for timeout in [15, 30, 60]:

    session_results.append({
        "timeout_minutes": timeout,
        "number_of_sessions": count_sessions(timeout)
    })


session_results_df = pd.DataFrame(session_results)

display(session_results_df)

NameError: name 'events_clean' is not defined

-> Đề xuất chọn 30 phút

Lý do:

1. 30 phút là một ngưỡng cân bằng

Nếu chọn 15 phút, khoảng 29.79% khoảng cách event lớn hơn timeout, dẫn đến việc tách session khá mạnh.

Nếu chọn 60 phút, chỉ 23.67% khoảng cách vượt timeout và số session giảm còn 1,726,713. Ngưỡng này có nguy cơ gộp những hành vi cách nhau khá xa vào cùng một session.

30 phút nằm ở giữa:

73.73% các khoảng cách event nằm trong cùng một session, trong khi các khoảng cách >30 phút được dùng để bắt đầu session mới.

2. Phù hợp với mục tiêu Sequential Recommendation

Nhóm đang nghiên cứu chuỗi:

view → view → addtocart → view → transaction

Nên session cần đủ dài để giữ được chuỗi hành vi có ý nghĩa, nhưng không quá dài đến mức gom những lần truy cập khác nhau vào cùng một phiên.

In [ ]:
# BƯỚC 26 — Tạo Session ID
# Session timeout = 30 phút

events_clean["time_diff"] = (
    events_clean
    .groupby("visitorid")["datetime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

events_clean["new_session"] = (
    events_clean["time_diff"].isna() |
    (events_clean["time_diff"] > 30)
)

events_clean["session_no"] = (
    events_clean
    .groupby("visitorid")["new_session"]
    .cumsum()
)

events_clean["session_id"] = (
    events_clean["visitorid"].astype(str)
    + "_"
    + events_clean["session_no"].astype(str)
)

print("Số session:", events_clean["session_id"].nunique())
print("Số user:", events_clean["visitorid"].nunique())

In [ ]:
# Kiểm tra một số session đầu tiên

display(
    events_clean[
        [
            "visitorid",
            "datetime",
            "event",
            "itemid",
            "time_diff",
            "session_no",
            "session_id"
        ]
    ].head(20)
)

In [ ]:
# Kiểm tra các khoảng cách trong cùng session

same_session_check = events_clean[
    (~events_clean["new_session"]) &
    (events_clean["time_diff"].notna())
]["time_diff"]

print("Khoảng cách lớn nhất giữa 2 event trong cùng session:",
      same_session_check.max(), "phút")

print("Có khoảng cách > 30 phút trong cùng session:",
      (same_session_check > 30).sum())

In [ ]:
# BƯỚC 27 — Phân tích độ dài Session

session_stats = (
    events_clean
    .groupby("session_id")
    .agg(
        visitorid=("visitorid", "first"),
        session_start=("datetime", "min"),
        session_end=("datetime", "max"),
        event_count=("event", "size"),
        unique_items=("itemid", "nunique")
    )
    .reset_index()
)

# Thời lượng session tính bằng phút
session_stats["session_duration_minutes"] = (
    session_stats["session_end"] -
    session_stats["session_start"]
).dt.total_seconds() / 60

print("Số session:", len(session_stats))
print("\nThống kê độ dài session:")
display(
    session_stats["event_count"].describe()
)

print("\nThống kê thời lượng session:")
display(
    session_stats["session_duration_minutes"].describe()
)

In [ ]:
# Phân phối số event trong mỗi session

session_length_distribution = (
    session_stats["event_count"]
    .value_counts()
    .sort_index()
    .head(20)
)

display(session_length_distribution)

In [ ]:
# 20 session có nhiều event nhất

display(
    session_stats
    .sort_values("event_count", ascending=False)
    .head(20)
)

In [ ]:
# BƯỚC 28 — Phân tích session có phát sinh transaction

session_purchase = (
    events_clean
    .groupby("session_id")
    .agg(
        event_count=("event", "size"),
        has_view=("event", lambda x: (x == "view").any()),
        has_addtocart=("event", lambda x: (x == "addtocart").any()),
        has_transaction=("event", lambda x: (x == "transaction").any()),
        transaction_count=("event", lambda x: (x == "transaction").sum())
    )
    .reset_index()
)

print("Tổng số session:", len(session_purchase))

print("\nSố session có transaction:")
print(
    session_purchase["has_transaction"].value_counts()
)

print("\nTỷ lệ session có transaction:")
print(
    session_purchase["has_transaction"].mean() * 100,
    "%"
)

print("\nSố transaction event theo session:")
display(
    session_purchase["transaction_count"].describe()
)

In [ ]:
# Phân phối loại session theo hành vi

def classify_session(row):
    if row["has_transaction"]:
        return "transaction"
    elif row["has_addtocart"]:
        return "addtocart"
    else:
        return "view_only"

session_purchase["session_type"] = (
    session_purchase.apply(classify_session, axis=1)
)

display(
    session_purchase["session_type"]
    .value_counts()
    .to_frame("session_count")
)

In [ ]:
# Tỷ lệ các loại session

session_type_summary = (
    session_purchase["session_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

display(session_type_summary)

NameError: name 'session_purchase' is not defined

In [ ]:
# BƯỚC 29 — Phân tích transaction trong các session có mua hàng

transaction_sessions = events_clean[
    events_clean["event"] == "transaction"
].copy()

print("Số transaction event:", len(transaction_sessions))

print("Số transaction ID duy nhất:",
      transaction_sessions["transactionid"].nunique())

print("Số session có transaction:",
      transaction_sessions["session_id"].nunique())

print("Số user có transaction:",
      transaction_sessions["visitorid"].nunique())

NameError: name 'events_clean' is not defined

Điều này cho thấy:

22.456 transaction events nhưng chỉ có 17.671 transaction ID → một đơn hàng có thể liên quan đến nhiều sản phẩm/item.
17.671 transaction ID nhưng chỉ có 14.296 session → một session có thể chứa nhiều transaction ID.
14.296 session mua hàng nhưng chỉ có 11.718 user → một user có thể mua hàng trong nhiều session khác nhau.
Trung bình mỗi session có transaction chứa khoảng 1,57 transaction event.
Phần lớn session mua hàng chỉ có 1 transaction event; một số session có rất nhiều transaction event, cao nhất 59.

In [ ]:
# Phân phối số transaction event trong mỗi session có transaction

transaction_per_session = (
    transaction_sessions
    .groupby("session_id")
    .size()
)

print("Thống kê transaction event / session có mua hàng:")

display(
    transaction_per_session.describe()
)

In [ ]:
# Những session có nhiều transaction event nhất

display(
    transaction_per_session
    .sort_values(ascending=False)
    .head(20)
)

NameError: name 'transaction_per_session' is not defined

In [ ]:
# BƯỚC 30 — Kiểm tra cấu trúc transaction

transaction_data = events_clean[
    events_clean["event"] == "transaction"
].copy()

transaction_stats = (
    transaction_data
    .groupby("transactionid")
    .agg(
        visitorid=("visitorid", "first"),
        session_count=("session_id", "nunique"),
        item_count=("itemid", "size"),
        unique_items=("itemid", "nunique")
    )
    .reset_index()
)

print("Số transaction ID:", len(transaction_stats))

print("\nThống kê số item / transaction:")
display(
    transaction_stats["item_count"].describe()
)

print("\nThống kê số item khác nhau / transaction:")
display(
    transaction_stats["unique_items"].describe()
)

print("\nThống kê số session / transaction:")
display(
    transaction_stats["session_count"].describe()
)

NameError: name 'events_clean' is not defined

In [ ]:
# 20 transaction có nhiều item nhất

display(
    transaction_stats
    .sort_values("item_count", ascending=False)
    .head(20)
)

NameError: name 'transaction_stats' is not defined

In [ ]:
# Kiểm tra một transaction ID có thuộc nhiều user hay không

transaction_user_check = (
    transaction_data
    .groupby("transactionid")["visitorid"]
    .nunique()
)

print(
    "Transaction ID thuộc nhiều hơn 1 user:",
    (transaction_user_check > 1).sum()
)

print(
    "Tỷ lệ:",
    (transaction_user_check > 1).mean() * 100,
    "%"
)

NameError: name 'transaction_data' is not defined

In [ ]:
# BƯỚC 31 — Kiểm tra item lặp trong cùng transaction

transaction_item_counts = (
    transaction_data
    .groupby(["transactionid", "itemid"])
    .size()
    .reset_index(name="event_count")
)

duplicate_transaction_items = transaction_item_counts[
    transaction_item_counts["event_count"] > 1
]

print(
    "Số cặp transaction-item bị lặp:",
    len(duplicate_transaction_items)
)

print(
    "Tổng số transaction-item duy nhất:",
    len(transaction_item_counts)
)

print(
    "Tỷ lệ transaction-item bị lặp:",
    len(duplicate_transaction_items) /
    len(transaction_item_counts) * 100,
    "%"
)

In [ ]:
# Xem các transaction-item bị lặp nhiều nhất

display(
    duplicate_transaction_items
    .sort_values("event_count", ascending=False)
    .head(20)
)

In [ ]:
# Kiểm tra số transaction có item bị lặp

transactions_with_duplicate_items = (
    duplicate_transaction_items["transactionid"]
    .nunique()
)

print(
    "Số transaction có item bị lặp:",
    transactions_with_duplicate_items
)

print(
    "Tỷ lệ:",
    transactions_with_duplicate_items /
    transaction_data["transactionid"].nunique() * 100,
    "%"
)

In [ ]:
# BƯỚC 32 — Kiểm tra hành vi trước transaction trong cùng session

session_event_sequence = (
    events_clean
    .sort_values(["session_id", "datetime", "timestamp"])
    .groupby("session_id")["event"]
    .apply(list)
)

transaction_sessions_sequence = (
    session_event_sequence[
        session_event_sequence.apply(lambda x: "transaction" in x)
    ]
)

print(
    "Số session có transaction:",
    len(transaction_sessions_sequence)
)

print("\nMột số sequence mẫu:")

for seq in transaction_sessions_sequence.head(20):
    print(seq)

In [ ]:
# Transaction có phải event cuối cùng trong session không?

transaction_position = (
    transaction_sessions_sequence
    .apply(lambda x: x[-1] == "transaction")
)

print(
    "Session có transaction là event cuối:",
    transaction_position.sum()
)

print(
    "Session có transaction không phải event cuối:",
    (~transaction_position).sum()
)

print(
    "Tỷ lệ transaction là event cuối:",
    transaction_position.mean() * 100,
    "%"
)

In [ ]:
# Kiểm tra event nào xuất hiện trước transaction trong session

def events_before_transaction(seq):
    if "transaction" not in seq:
        return None

    first_transaction = seq.index("transaction")
    return seq[:first_transaction]

before_transaction = (
    transaction_sessions_sequence
    .apply(events_before_transaction)
)

print("Một số hành vi trước transaction:")

for seq in before_transaction.head(20):
    print(seq)

In [ ]:
# BƯỚC 33 — Tạo Clean Event Dataset chính thức

clean_events_final = events_clean[
    [
        "timestamp",
        "datetime",
        "visitorid",
        "session_id",
        "event",
        "itemid",
        "transactionid"
    ]
].copy()

print("Shape:", clean_events_final.shape)

print("\nColumns:")
print(clean_events_final.columns.tolist())

print("\nMissing values:")
display(clean_events_final.isna().sum())

print("\nEvent distribution:")
display(clean_events_final["event"].value_counts())

NameError: name 'events_clean' is not defined

In [ ]:
# Validation cuối Phase C

print("Duplicate rows:", clean_events_final.duplicated().sum())

print(
    "Users:",
    clean_events_final["visitorid"].nunique()
)

print(
    "Sessions:",
    clean_events_final["session_id"].nunique()
)

print(
    "Items:",
    clean_events_final["itemid"].nunique()
)

print(
    "Transactions:",
    clean_events_final["transactionid"].nunique()
)

print(
    "Datetime min:",
    clean_events_final["datetime"].min()
)

print(
    "Datetime max:",
    clean_events_final["datetime"].max()
)

NameError: name 'clean_events_final' is not defined

In [ ]:
# BƯỚC 34 — Export Clean Event Dataset

clean_events_path = "/content/clean_events_final.csv"

clean_events_final.to_csv(
    clean_events_path,
    index=False
)

print("Đã xuất file thành công!")
print("File:", clean_events_path)

NameError: name 'clean_events_final' is not defined

In [ ]:
import os

print("File tồn tại:", os.path.exists(clean_events_path))

if os.path.exists(clean_events_path):
    file_size_mb = os.path.getsize(clean_events_path) / (1024 ** 2)
    print(f"Kích thước file: {file_size_mb:.2f} MB")

File tồn tại: False


In [ ]:
# BƯỚC 35 — Kiểm tra User–Item–Event

user_item_event = clean_events_final[
    [
        "visitorid",
        "session_id",
        "datetime",
        "event",
        "itemid",
        "transactionid"
    ]
].copy()

print("Shape:", user_item_event.shape)

print("\nSố user:", user_item_event["visitorid"].nunique())
print("Số item:", user_item_event["itemid"].nunique())
print("Số session:", user_item_event["session_id"].nunique())

print("\nPhân phối event:")
display(
    user_item_event["event"].value_counts()
)

print("\n5 dòng đầu:")
display(
    user_item_event.head()
)

NameError: name 'clean_events_final' is not defined

In [ ]:
# Kiểm tra các tổ hợp User–Item–Event

uie_duplicate = (
    user_item_event
    .groupby(
        ["visitorid", "itemid", "event"]
    )
    .size()
    .reset_index(name="event_count")
)

print(
    "Số User–Item–Event duy nhất:",
    len(uie_duplicate)
)

print(
    "Số User–Item–Event xuất hiện nhiều lần:",
    (uie_duplicate["event_count"] > 1).sum()
)

print(
    "Tỷ lệ:",
    (uie_duplicate["event_count"] > 1).mean() * 100,
    "%"
)

NameError: name 'user_item_event' is not defined

In [ ]:
# BƯỚC 36 — Tạo User–Item–Event Table chính thức

user_item_event_final = (
    clean_events_final[
        [
            "visitorid",
            "session_id",
            "datetime",
            "event",
            "itemid",
            "transactionid"
        ]
    ]
    .sort_values(
        by=["visitorid", "datetime"]
    )
    .reset_index(drop=True)
)

print("Shape:", user_item_event_final.shape)

print("\nColumns:")
print(user_item_event_final.columns.tolist())

print("\n5 dòng đầu:")
display(user_item_event_final.head())

print("\n5 dòng cuối:")
display(user_item_event_final.tail())

NameError: name 'clean_events_final' is not defined

In [ ]:
# Kiểm tra thứ tự thời gian trong từng user

user_time_diff = (
    user_item_event_final
    .groupby("visitorid")["datetime"]
    .diff()
)

negative_time_diff = (
    user_time_diff < pd.Timedelta(0)
).sum()

print(
    "Số trường hợp thời gian giảm trong cùng user:",
    negative_time_diff
)

print(
    "Thời gian giảm:",
    "Không có" if negative_time_diff == 0 else "Có"
)

NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 37 — Kiểm tra cấu trúc User–Item–Event Table

print("===== KIỂM TRA USER–ITEM–EVENT TABLE =====")

# 1. Missing values
print("\n1. Missing values:")
print(user_item_event_final.isna().sum())

# 2. Số user
print("\n2. Unique users:")
print(user_item_event_final["visitorid"].nunique())

# 3. Số session
print("\n3. Unique sessions:")
print(user_item_event_final["session_id"].nunique())

# 4. Số item
print("\n4. Unique items:")
print(user_item_event_final["itemid"].nunique())

# 5. Số transaction
print("\n5. Unique transactions:")
print(
    user_item_event_final["transactionid"].dropna().nunique()
)

# 6. Phân bố event
print("\n6. Event distribution:")
print(user_item_event_final["event"].value_counts())

# 7. Kiểm tra duplicate toàn dòng
print("\n7. Duplicate rows:")
print(user_item_event_final.duplicated().sum())

===== KIỂM TRA USER–ITEM–EVENT TABLE =====

1. Missing values:


NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 38 — Kiểm tra quan hệ User – Session – Transaction

print("===== KIỂM TRA QUAN HỆ USER – SESSION – TRANSACTION =====")

# 1. Mỗi session thuộc bao nhiêu user?
session_user_count = (
    user_item_event_final
    .groupby("session_id")["visitorid"]
    .nunique()
)

print("\n1. Session thuộc nhiều user:")
print((session_user_count > 1).sum())

# 2. Một user có bao nhiêu session?
user_session_count = (
    user_item_event_final
    .groupby("visitorid")["session_id"]
    .nunique()
)

print("\n2. Số session mỗi user:")
print(user_session_count.describe())

# 3. Một transaction thuộc bao nhiêu user?
transaction_user_count = (
    user_item_event_final[
        user_item_event_final["transactionid"].notna()
    ]
    .groupby("transactionid")["visitorid"]
    .nunique()
)

print("\n3. Transaction thuộc nhiều user:")
print((transaction_user_count > 1).sum())

# 4. Một transaction thuộc bao nhiêu session?
transaction_session_count = (
    user_item_event_final[
        user_item_event_final["transactionid"].notna()
    ]
    .groupby("transactionid")["session_id"]
    .nunique()
)

print("\n4. Transaction xuất hiện ở nhiều session:")
print((transaction_session_count > 1).sum())

===== KIỂM TRA QUAN HỆ USER – SESSION – TRANSACTION =====


NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 39 — Kiểm tra 7 transaction xuất hiện ở nhiều session

multi_session_transactions = (
    transaction_session_count[
        transaction_session_count > 1
    ]
    .index
)

print("Số transaction xuất hiện ở nhiều session:",
      len(multi_session_transactions))

print("\nDanh sách transaction:")
print(multi_session_transactions.tolist())

print("\nChi tiết:")

transaction_multi_session_detail = (
    user_item_event_final[
        user_item_event_final["transactionid"].isin(
            multi_session_transactions
        )
    ]
    .groupby("transactionid")
    .agg(
        visitorid=("visitorid", "nunique"),
        sessions=("session_id", "nunique"),
        events=("event", "count"),
        items=("itemid", "nunique"),
        first_time=("datetime", "min"),
        last_time=("datetime", "max")
    )
    .reset_index()
)

display(transaction_multi_session_detail)

Kết luận BƯỚC 39

Không xóa 7 transaction này.

Lý do:

Một transactionid có thể xuất hiện ở nhiều session của cùng một user trong dữ liệu thực tế. Việc này không làm sai cấu trúc User–Item–Event Table và không phải duplicate toàn dòng.

Đặc biệt, nếu xóa chúng thì sẽ làm mất event giao dịch hợp lệ.


In [ ]:
# BƯỚC 40 — Kiểm tra mức độ lặp lại User–Item theo từng Event

print("===== USER–ITEM–EVENT REPEAT ANALYSIS =====")

# Đếm số lần mỗi user thực hiện từng event trên từng item
user_item_event_count = (
    user_item_event_final
    .groupby(
        ["visitorid", "itemid", "event"]
    )
    .size()
    .reset_index(name="event_count")
)

print("\n1. Số User–Item–Event combinations:")
print(len(user_item_event_count))

# Phân bố số lần thực hiện
print("\n2. Phân bố số lần lặp:")
print(
    user_item_event_count["event_count"]
    .describe()
)

# Số combination chỉ xuất hiện 1 lần
single_event = (
    user_item_event_count["event_count"] == 1
).sum()

# Số combination xuất hiện nhiều hơn 1 lần
repeat_event = (
    user_item_event_count["event_count"] > 1
).sum()

print("\n3. Chỉ xuất hiện 1 lần:")
print(single_event)

print("\n4. Xuất hiện nhiều hơn 1 lần:")
print(repeat_event)

print(
    "\n5. Tỷ lệ User–Item–Event bị lặp:",
    round(
        repeat_event / len(user_item_event_count) * 100,
        2
    ),
    "%"
)

# Phân tích riêng theo event
print("\n6. Phân tích theo từng Event:")

event_repeat_summary = (
    user_item_event_count
    .groupby("event")
    .agg(
        unique_user_item_event=("event_count", "size"),
        repeated_user_item_event=(
            "event_count",
            lambda x: (x > 1).sum()
        ),
        max_repeat=("event_count", "max")
    )
    .reset_index()
)

event_repeat_summary["repeat_rate_%"] = (
    event_repeat_summary["repeated_user_item_event"]
    / event_repeat_summary["unique_user_item_event"]
    * 100
)

display(event_repeat_summary)

===== USER–ITEM–EVENT REPEAT ANALYSIS =====


NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 41 — Phân tích quan hệ User–Item

print("===== USER–ITEM RELATIONSHIP ANALYSIS =====")

# 1. Số item khác nhau mà mỗi user tương tác
user_item_count = (
    user_item_event_final
    .groupby("visitorid")["itemid"]
    .nunique()
)

print("\n1. Số item khác nhau mỗi user:")
print(user_item_count.describe())

# 2. Số user khác nhau tương tác với mỗi item
item_user_count = (
    user_item_event_final
    .groupby("itemid")["visitorid"]
    .nunique()
)

print("\n2. Số user khác nhau mỗi item:")
print(item_user_count.describe())

# 3. User chỉ tương tác với 1 item
users_one_item = (
    user_item_count == 1
).sum()

print("\n3. User chỉ tương tác với 1 item:")
print(users_one_item)

# 4. Item chỉ được 1 user tương tác
items_one_user = (
    item_user_count == 1
).sum()

print("\n4. Item chỉ được 1 user tương tác:")
print(items_one_user)

# 5. Top 10 user có nhiều item tương tác nhất
print("\n5. Top 10 user có nhiều item tương tác nhất:")

top_users = (
    user_item_count
    .sort_values(ascending=False)
    .head(10)
)

display(top_users)

# 6. Top 10 item có nhiều user tương tác nhất
print("\n6. Top 10 item có nhiều user tương tác nhất:")

top_items = (
    item_user_count
    .sort_values(ascending=False)
    .head(10)
)

display(top_items)

In [ ]:
# BƯỚC 42 — Kiểm tra độ phủ hành vi trên User–Item

print("===== USER–ITEM BEHAVIOR COVERAGE =====")

# Lấy các loại event khác nhau của mỗi User–Item
user_item_behavior = (
    user_item_event_final
    .groupby(["visitorid", "itemid"])["event"]
    .agg(lambda x: set(x))
    .reset_index(name="event_set")
)

# Chuyển tập event thành nhãn dễ đọc
def classify_behavior(event_set):
    if event_set == {"view"}:
        return "view_only"
    elif event_set == {"view", "addtocart"}:
        return "view_addtocart"
    elif event_set == {"view", "transaction"}:
        return "view_transaction"
    elif event_set == {"addtocart"}:
        return "addtocart_only"
    elif event_set == {"addtocart", "transaction"}:
        return "addtocart_transaction"
    elif event_set == {"transaction"}:
        return "transaction_only"
    elif event_set == {"view", "addtocart", "transaction"}:
        return "view_addtocart_transaction"
    else:
        return "other"

user_item_behavior["behavior_type"] = (
    user_item_behavior["event_set"]
    .apply(classify_behavior)
)

print("\n1. Tổng số User–Item:")
print(len(user_item_behavior))

# Phân bố các loại hành vi
print("\n2. Phân bố Behavior Type:")

behavior_distribution = (
    user_item_behavior["behavior_type"]
    .value_counts()
    .to_frame("user_item_count")
)

behavior_distribution["percentage_%"] = (
    behavior_distribution["user_item_count"]
    / len(user_item_behavior)
    * 100
)

display(behavior_distribution)

# Kiểm tra User–Item có transaction
print("\n3. User–Item có transaction:")
print(
    user_item_behavior["behavior_type"]
    .str.contains("transaction")
    .sum()
)

# Kiểm tra User–Item có addtocart
print("\n4. User–Item có addtocart:")
print(
    user_item_behavior["behavior_type"]
    .str.contains("addtocart")
    .sum()
)

# Kiểm tra User–Item chỉ view
print("\n5. User–Item chỉ view:")
print(
    (
        user_item_behavior["behavior_type"]
        == "view_only"
    ).sum()
)

===== USER–ITEM BEHAVIOR COVERAGE =====


NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 43 — Kiểm tra tính nhất quán Event – Transaction ID

print("===== EVENT – TRANSACTION CONSISTENCY =====")

# 1. Transaction event nhưng thiếu transactionid
transaction_missing_id = (
    (user_item_event_final["event"] == "transaction") &
    (user_item_event_final["transactionid"].isna())
).sum()

print("\n1. Transaction nhưng thiếu transactionid:")
print(transaction_missing_id)

# 2. View có transactionid
view_with_transaction = (
    (user_item_event_final["event"] == "view") &
    (user_item_event_final["transactionid"].notna())
).sum()

print("\n2. View có transactionid:")
print(view_with_transaction)

# 3. Addtocart có transactionid
addtocart_with_transaction = (
    (user_item_event_final["event"] == "addtocart") &
    (user_item_event_final["transactionid"].notna())
).sum()

print("\n3. Addtocart có transactionid:")
print(addtocart_with_transaction)

# 4. Kiểm tra toàn bộ event ngoài 3 loại hợp lệ
invalid_events = (
    ~user_item_event_final["event"]
    .isin(["view", "addtocart", "transaction"])
).sum()

print("\n4. Event không hợp lệ:")
print(invalid_events)

# 5. Tổng số transaction ID hợp lệ
print("\n5. Unique transaction IDs:")
print(
    user_item_event_final["transactionid"]
    .dropna()
    .nunique()
)

In [ ]:
# BƯỚC 44 — Đối chiếu User–Item–Event Table với Clean Event Dataset

print("===== FINAL INTEGRITY CHECK — PHASE D =====")

# 1. So sánh số dòng
print("\n1. Số dòng:")
print("clean_events_final:", len(clean_events_final))
print("user_item_event_final:", len(user_item_event_final))

# 2. So sánh số event theo loại
print("\n2. So sánh event counts:")

clean_event_counts = (
    clean_events_final["event"]
    .value_counts()
    .sort_index()
)

uie_event_counts = (
    user_item_event_final["event"]
    .value_counts()
    .sort_index()
)

comparison = pd.DataFrame({
    "clean_events_final": clean_event_counts,
    "user_item_event_final": uie_event_counts
})

comparison["difference"] = (
    comparison["user_item_event_final"]
    - comparison["clean_events_final"]
)

display(comparison)

# 3. Kiểm tra số user
print("\n3. Unique users:")
print(
    clean_events_final["visitorid"].nunique(),
    "vs",
    user_item_event_final["visitorid"].nunique()
)

# 4. Kiểm tra số item
print("\n4. Unique items:")
print(
    clean_events_final["itemid"].nunique(),
    "vs",
    user_item_event_final["itemid"].nunique()
)

# 5. Kiểm tra số session
print("\n5. Unique sessions:")
print(
    clean_events_final["session_id"].nunique(),
    "vs",
    user_item_event_final["session_id"].nunique()
)

# 6. Kiểm tra số transaction
print("\n6. Unique transactions:")
print(
    clean_events_final["transactionid"].dropna().nunique(),
    "vs",
    user_item_event_final["transactionid"].dropna().nunique()
)

===== FINAL INTEGRITY CHECK — PHASE D =====

1. Số dòng:


NameError: name 'clean_events_final' is not defined

In [ ]:
# BƯỚC 45 — Export User–Item–Event Table

user_item_event_path = "/content/user_item_event_final.csv"

user_item_event_final.to_csv(
    user_item_event_path,
    index=False
)

print("Đã xuất file thành công!")
print("File:", user_item_event_path)

# Kiểm tra file
import os

print("\nFile tồn tại:", os.path.exists(user_item_event_path))

if os.path.exists(user_item_event_path):
    file_size_mb = os.path.getsize(user_item_event_path) / (1024 ** 2)
    print(f"Kích thước file: {file_size_mb:.2f} MB")

NameError: name 'user_item_event_final' is not defined

In [ ]:
# BƯỚC 46 — Kiểm tra các file Item Metadata

print("===== PHASE E — ITEM METADATA INVENTORY =====")

# ==============================
# 1. ITEM PROPERTIES PART 1
# ==============================

print("\n===== item_properties_part1.csv =====")

item_prop_1 = pd.read_csv(
    "/content/data/item_properties_part1.csv"
)

print("Shape:", item_prop_1.shape)

print("\nColumns:")
print(item_prop_1.columns.tolist())

print("\n5 dòng đầu:")
display(item_prop_1.head())

print("\nMissing values:")
print(item_prop_1.isna().sum())

print("\nUnique:")
print("Items:", item_prop_1["itemid"].nunique())
print("Properties:", item_prop_1["property"].nunique())
print("Values:", item_prop_1["value"].nunique())


# ==============================
# 2. ITEM PROPERTIES PART 2
# ==============================

print("\n===== item_properties_part2.csv =====")

item_prop_2 = pd.read_csv(
    "/content/data/item_properties_part2.csv"
)

print("Shape:", item_prop_2.shape)

print("\nColumns:")
print(item_prop_2.columns.tolist())

print("\n5 dòng đầu:")
display(item_prop_2.head())

print("\nMissing values:")
print(item_prop_2.isna().sum())

print("\nUnique:")
print("Items:", item_prop_2["itemid"].nunique())
print("Properties:", item_prop_2["property"].nunique())
print("Values:", item_prop_2["value"].nunique())


# ==============================
# 3. CATEGORY TREE
# ==============================

print("\n===== category_tree.csv =====")

category_tree = pd.read_csv(
    "/content/data/category_tree.csv"
)

print("Shape:", category_tree.shape)

print("\nColumns:")
print(category_tree.columns.tolist())

print("\n5 dòng đầu:")
display(category_tree.head())

print("\nMissing values:")
print(category_tree.isna().sum())

print("\nUnique:")
print("Categories:", category_tree["categoryid"].nunique())
print("Parent categories:", category_tree["parentid"].nunique())

In [ ]:
# BƯỚC 47 — Kiểm tra chồng lấn Item Properties Part 1 và Part 2

print("===== OVERLAP CHECK: PART 1 vs PART 2 =====")

# ==========================================
# 1. Khoảng thời gian
# ==========================================

item_prop_1["datetime"] = pd.to_datetime(
    item_prop_1["timestamp"],
    unit="ms"
)

item_prop_2["datetime"] = pd.to_datetime(
    item_prop_2["timestamp"],
    unit="ms"
)

print("\n1. Khoảng thời gian:")

print(
    "Part 1:",
    item_prop_1["datetime"].min(),
    "→",
    item_prop_1["datetime"].max()
)

print(
    "Part 2:",
    item_prop_2["datetime"].min(),
    "→",
    item_prop_2["datetime"].max()
)


# ==========================================
# 2. Exact duplicate trong từng file
# ==========================================

print("\n2. Duplicate hoàn toàn trong từng file:")

print(
    "Part 1:",
    item_prop_1[
        ["timestamp", "itemid", "property", "value"]
    ].duplicated().sum()
)

print(
    "Part 2:",
    item_prop_2[
        ["timestamp", "itemid", "property", "value"]
    ].duplicated().sum()
)


# ==========================================
# 3. Exact duplicate giữa Part 1 và Part 2
# ==========================================

print("\n3. Exact duplicate giữa Part 1 và Part 2:")

part1_keys = item_prop_1[
    ["timestamp", "itemid", "property", "value"]
].drop_duplicates()

part2_keys = item_prop_2[
    ["timestamp", "itemid", "property", "value"]
].drop_duplicates()

cross_overlap = part1_keys.merge(
    part2_keys,
    on=["timestamp", "itemid", "property", "value"],
    how="inner"
)

print(
    "Số bản ghi trùng hoàn toàn giữa 2 file:",
    len(cross_overlap)
)


# ==========================================
# 4. Chồng lấn Item–Property
# ==========================================

print("\n4. Item–Property xuất hiện ở cả 2 file:")

part1_item_property = (
    item_prop_1[
        ["itemid", "property"]
    ]
    .drop_duplicates()
)

part2_item_property = (
    item_prop_2[
        ["itemid", "property"]
    ]
    .drop_duplicates()
)

item_property_overlap = part1_item_property.merge(
    part2_item_property,
    on=["itemid", "property"],
    how="inner"
)

print(
    "Số Item–Property xuất hiện ở cả 2 file:",
    len(item_property_overlap)
)


# ==========================================
# 5. Chồng lấn item
# ==========================================

print("\n5. Item xuất hiện ở cả 2 file:")

common_items = set(
    item_prop_1["itemid"]
).intersection(
    set(item_prop_2["itemid"])
)

print(
    "Số item xuất hiện ở cả 2 file:",
    len(common_items)
)

print(
    "Tổng item Part 1:",
    item_prop_1["itemid"].nunique()
)

print(
    "Tổng item Part 2:",
    item_prop_2["itemid"].nunique()
)

===== OVERLAP CHECK: PART 1 vs PART 2 =====


NameError: name 'item_prop_1' is not defined

In [ ]:
# BƯỚC 48 — Kiểm tra lịch sử thay đổi Item–Property

print("===== ITEM–PROPERTY HISTORY CHECK =====")

# Gộp tạm 2 Part để kiểm tra lịch sử
item_properties_all = pd.concat(
    [
        item_prop_1[
            ["timestamp", "datetime", "itemid", "property", "value"]
        ],
        item_prop_2[
            ["timestamp", "datetime", "itemid", "property", "value"]
        ]
    ],
    ignore_index=True
)

print("\n1. Tổng số bản ghi sau khi ghép:")
print("Số dòng:", len(item_properties_all))

print("\n2. Số Item:")
print("Unique item:", item_properties_all["itemid"].nunique())

print("\n3. Số Property:")
print("Unique property:", item_properties_all["property"].nunique())

# Số lần xuất hiện của mỗi Item–Property
item_property_history = (
    item_properties_all
    .groupby(["itemid", "property"])
    .size()
)

print("\n4. Số lần xuất hiện của mỗi Item–Property:")
print(item_property_history.describe())

# Item–Property xuất hiện nhiều hơn 1 lần
repeated_item_property = (
    item_property_history[
        item_property_history > 1
    ]
)

print("\n5. Item–Property xuất hiện nhiều hơn 1 lần:")
print("Số Item–Property:", len(repeated_item_property))
print(
    "Tỷ lệ:",
    round(
        len(repeated_item_property)
        / len(item_property_history)
        * 100,
        2
    ),
    "%"
)

# Kiểm tra Item–Property có nhiều giá trị khác nhau
item_property_values = (
    item_properties_all
    .groupby(["itemid", "property"])["value"]
    .nunique()
)

multiple_values = (
    item_property_values[
        item_property_values > 1
    ]
)

print("\n6. Item–Property có nhiều giá trị khác nhau:")
print("Số Item–Property:", len(multiple_values))
print(
    "Tỷ lệ:",
    round(
        len(multiple_values)
        / len(item_property_values)
        * 100,
        2
    ),
    "%"
)

# Kiểm tra cùng timestamp nhưng khác value
same_timestamp_diff_value = (
    item_properties_all
    .groupby(["timestamp", "itemid", "property"])["value"]
    .nunique()
)

conflict_timestamp = (
    same_timestamp_diff_value[
        same_timestamp_diff_value > 1
    ]
)

print("\n7. Cùng timestamp + item + property nhưng khác value:")
print("Số trường hợp:", len(conflict_timestamp))

# Một vài ví dụ Item–Property có nhiều giá trị
print("\n8. Ví dụ Item–Property có nhiều giá trị:")

examples = (
    item_properties_all[
        item_properties_all.set_index(
            ["itemid", "property"]
        ).index.isin(
            multiple_values.head(5).index
        )
    ]
    .sort_values(
        by=["itemid", "property", "timestamp"]
    )
)

print(
    examples[
        ["datetime", "itemid", "property", "value"]
    ].head(20)
)

In [ ]:
# BƯỚC 49A — Kiểm tra lại môi trường Colab

import pandas as pd
import os

print("===== KIỂM TRA MÔI TRƯỜNG =====")

print("\nPandas version:")
print(pd.__version__)

print("\nKiểm tra biến dữ liệu:")

print(
    "item_prop_1:",
    "ĐÃ CÓ" if "item_prop_1" in globals() else "CHƯA CÓ"
)

print(
    "item_prop_2:",
    "ĐÃ CÓ" if "item_prop_2" in globals() else "CHƯA CÓ"
)

print(
    "category_tree:",
    "ĐÃ CÓ" if "category_tree" in globals() else "CHƯA CÓ"
)

print(
    "clean_events_final:",
    "ĐÃ CÓ" if "clean_events_final" in globals() else "CHƯA CÓ"
)

print(
    "user_item_event_final:",
    "ĐÃ CÓ" if "user_item_event_final" in globals() else "CHƯA CÓ"
)

print("\nKiểm tra thư mục /content/data:")

if os.path.exists("/content/data"):
    print("Thư mục /content/data: CÓ")
    print(os.listdir("/content/data"))
else:
    print("Thư mục /content/data: KHÔNG CÓ")

===== KIỂM TRA MÔI TRƯỜNG =====

Pandas version:
2.2.3

Kiểm tra biến dữ liệu:
item_prop_1: CHƯA CÓ
item_prop_2: CHƯA CÓ
category_tree: CHƯA CÓ
clean_events_final: CHƯA CÓ
user_item_event_final: CHƯA CÓ

Kiểm tra thư mục /content/data:
Thư mục /content/data: KHÔNG CÓ


In [ ]:
# BƯỚC 49B — Khôi phục dữ liệu Phase E sau khi runtime reset

import pandas as pd

print("===== KHÔI PHỤC DỮ LIỆU PHASE E =====")

# 1. Load Item Properties Part 1
item_prop_1 = pd.read_csv(
    "/content/data/item_properties_part1.csv"
)

# 2. Load Item Properties Part 2
item_prop_2 = pd.read_csv(
    "/content/data/item_properties_part2.csv"
)

# 3. Load Category Tree
category_tree = pd.read_csv(
    "/content/data/category_tree.csv"
)

print("\nĐã load dữ liệu thành công!")

print("\nPart 1:")
print("Shape:", item_prop_1.shape)

print("\nPart 2:")
print("Shape:", item_prop_2.shape)

print("\nCategory Tree:")
print("Shape:", category_tree.shape)

print("\nColumns Part 1:")
print(item_prop_1.columns.tolist())

print("\nColumns Part 2:")
print(item_prop_2.columns.tolist())

print("\nColumns Category Tree:")
print(category_tree.columns.tolist())

===== KHÔI PHỤC DỮ LIỆU PHASE E =====


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/item_properties_part1.csv'

In [ ]:
# BƯỚC 49C — Kiểm tra tính nhất quán lịch sử Item–Property

print("===== ITEM–PROPERTY TEMPORAL CONSISTENCY =====")

# ==================================================
# 1. Ghép Part 1 + Part 2
# ==================================================

item_properties_all = pd.concat(
    [
        item_prop_1,
        item_prop_2
    ],
    ignore_index=True
)

print("\n1. Tổng số bản ghi:")
print(len(item_properties_all))

# ==================================================
# 2. Chuyển timestamp sang datetime
# ==================================================

item_properties_all["datetime"] = pd.to_datetime(
    item_properties_all["timestamp"],
    unit="ms"
)

# ==================================================
# 3. Sắp xếp theo Item → Property → Timestamp
# ==================================================

item_properties_all = (
    item_properties_all
    .sort_values(
        by=["itemid", "property", "timestamp"]
    )
    .reset_index(drop=True)
)

# ==================================================
# 4. Kiểm tra timestamp trùng
# ==================================================

same_timestamp = (
    item_properties_all
    .groupby(
        ["itemid", "property", "timestamp"]
    )
    .size()
)

duplicate_timestamp = same_timestamp[
    same_timestamp > 1
]

print("\n2. Timestamp trùng trong cùng Item–Property:")
print(
    "Số trường hợp:",
    len(duplicate_timestamp)
)

# ==================================================
# 5. Kiểm tra timestamp bị giảm
# ==================================================

time_diff_property = (
    item_properties_all
    .groupby(
        ["itemid", "property"]
    )["timestamp"]
    .diff()
)

negative_time_diff = time_diff_property[
    time_diff_property < 0
]

print("\n3. Timestamp bị giảm/ngược thời gian:")
print(
    "Số trường hợp:",
    len(negative_time_diff)
)

# ==================================================
# 6. Xác định value có thay đổi hay không
# ==================================================

previous_value = (
    item_properties_all
    .groupby(
        ["itemid", "property"]
    )["value"]
    .shift()
)

item_properties_all["value_changed"] = (
    item_properties_all["value"]
    .ne(previous_value)
    .astype(int)
)

# Dòng đầu tiên của mỗi Item–Property
# không được tính là một lần thay đổi
first_record = (
    item_properties_all
    .groupby(
        ["itemid", "property"]
    )
    .cumcount()
    == 0
)

item_properties_all.loc[
    first_record,
    "value_changed"
] = 0

# ==================================================
# 7. Tổng số lần value thay đổi
# ==================================================

total_value_changes = int(
    item_properties_all["value_changed"].sum()
)

print("\n4. Tổng số lần value thay đổi:")
print(
    total_value_changes
)

# ==================================================
# 8. Item–Property có nhiều value khác nhau
# ==================================================

value_counts = (
    item_properties_all
    .groupby(
        ["itemid", "property"]
    )["value"]
    .nunique()
)

changed_pairs = value_counts[
    value_counts > 1
]

print("\n5. Item–Property có nhiều value khác nhau:")
print(
    "Số Item–Property:",
    len(changed_pairs)
)

# ==================================================
# 9. Phân phối số lần thay đổi
# ==================================================

change_count_per_pair = (
    item_properties_all
    .groupby(
        ["itemid", "property"]
    )["value_changed"]
    .sum()
)

positive_changes = change_count_per_pair[
    change_count_per_pair > 0
]

print("\n6. Phân phối số lần thay đổi:")
print(
    positive_changes.describe()
)

# ==================================================
# 10. Khoảng cách giữa các lần ghi nhận
# ==================================================

positive_time_diff = time_diff_property[
    time_diff_property > 0
]

print("\n7. Khoảng cách thời gian giữa các lần ghi nhận:")

print(
    positive_time_diff.describe()
)

# ==================================================
# 11. Ví dụ Item–Property có thay đổi value
# ==================================================

print("\n8. Ví dụ Item–Property có thay đổi value:")

example_pairs = changed_pairs.head(5).index

example_data = (
    item_properties_all
    .set_index(
        ["itemid", "property"]
    )
    .loc[
        example_pairs
    ]
    .reset_index()
)

print(
    example_data[
        [
            "datetime",
            "itemid",
            "property",
            "value"
        ]
    ].to_string(index=False)
)

# ==================================================
# 12. Kiểm tra cuối BƯỚC 49
# ==================================================

print("\n===== KIỂM TRA CUỐI BƯỚC 49 =====")

print(
    "Tổng số dòng:",
    len(item_properties_all)
)

print(
    "Unique Item:",
    item_properties_all["itemid"].nunique()
)

print(
    "Unique Property:",
    item_properties_all["property"].nunique()
)

===== ITEM–PROPERTY TEMPORAL CONSISTENCY =====


NameError: name 'item_prop_1' is not defined

In [ ]:
# BƯỚC 50 — Kiểm tra Property categoryid

print("===== CATEGORYID PROPERTY CHECK =====")

# 1. Số bản ghi có property = categoryid
category_property = item_properties_all[
    item_properties_all["property"] == "categoryid"
].copy()

print("\n1. Số bản ghi property = categoryid:")
print(len(category_property))

# 2. Số item có categoryid
print("\n2. Số item có categoryid:")
print(
    category_property["itemid"].nunique()
)

# 3. Số categoryid khác nhau
print("\n3. Số categoryid khác nhau:")
print(
    category_property["value"].nunique()
)

# 4. Một item có bao nhiêu categoryid khác nhau?
item_category_count = (
    category_property
    .groupby("itemid")["value"]
    .nunique()
)

print("\n4. Phân phối số categoryid / item:")
print(
    item_category_count.describe()
)

# 5. Item có nhiều categoryid
multiple_categories = item_category_count[
    item_category_count > 1
]

print("\n5. Item có nhiều categoryid:")
print(
    "Số item:",
    len(multiple_categories)
)

# 6. Kiểm tra categoryid có nằm trong category_tree không
category_tree_ids = set(
    category_tree["categoryid"]
)

property_category_ids = set(
    category_property["value"]
)

# Chuyển value sang string để kiểm tra cấu trúc trước
print("\n6. Kiểu dữ liệu value của categoryid:")
print(
    category_property["value"].dtype
)

print("\n7. Một số giá trị categoryid:")
print(
    category_property["value"]
    .drop_duplicates()
    .head(20)
    .tolist()
)

# 7. Kiểm tra giá trị rỗng
print("\n8. Missing categoryid:")
print(
    category_property["value"].isna().sum()
)

# 8. Category Tree
print("\n9. Category Tree:")
print(
    "Số category:",
    category_tree["categoryid"].nunique()
)

print(
    "Missing parentid:",
    category_tree["parentid"].isna().sum()
)

In [ ]:
# BƯỚC 51 — Kiểm tra lịch sử Category của Item

print("===== ITEM–CATEGORY TEMPORAL CHECK =====")

# --------------------------------------------------
# 1. Tách các bản ghi categoryid
# --------------------------------------------------

item_category_history = (
    item_properties_all[
        item_properties_all["property"] == "categoryid"
    ][
        [
            "timestamp",
            "datetime",
            "itemid",
            "value"
        ]
    ]
    .copy()
)

# Đổi tên value → categoryid
item_category_history = (
    item_category_history
    .rename(
        columns={
            "value": "categoryid"
        }
    )
)

# Chuyển categoryid sang số
item_category_history["categoryid"] = pd.to_numeric(
    item_category_history["categoryid"],
    errors="coerce"
)

print("\n1. Tổng số bản ghi:")
print(len(item_category_history))

print("\n2. Missing categoryid sau khi chuyển kiểu:")
print(
    item_category_history["categoryid"].isna().sum()
)

# --------------------------------------------------
# 2. Kiểm tra Item–Category bị lặp
# --------------------------------------------------

item_category_count = (
    item_category_history
    .groupby("itemid")["categoryid"]
    .nunique()
)

print("\n3. Phân phối số Category / Item:")
print(
    item_category_count.describe()
)

# --------------------------------------------------
# 3. Item có nhiều Category
# --------------------------------------------------

multiple_category_items = item_category_count[
    item_category_count > 1
]

print("\n4. Item có nhiều Category:")
print(
    "Số item:",
    len(multiple_category_items)
)

print(
    "Tỷ lệ:",
    round(
        len(multiple_category_items)
        / len(item_category_count)
        * 100,
        2
    ),
    "%"
)

# --------------------------------------------------
# 4. Kiểm tra Category có thay đổi theo thời gian
# --------------------------------------------------

item_category_history = (
    item_category_history
    .sort_values(
        by=["itemid", "timestamp"]
    )
    .reset_index(drop=True)
)

previous_category = (
    item_category_history
    .groupby("itemid")["categoryid"]
    .shift()
)

item_category_history["category_changed"] = (
    item_category_history["categoryid"]
    .ne(previous_category)
    .astype(int)
)

# Không tính bản ghi đầu tiên của mỗi item
first_item_record = (
    item_category_history
    .groupby("itemid")
    .cumcount()
    == 0
)

item_category_history.loc[
    first_item_record,
    "category_changed"
] = 0

print("\n5. Tổng số lần Category thay đổi:")
print(
    int(
        item_category_history[
            "category_changed"
        ].sum()
    )
)

# --------------------------------------------------
# 5. Kiểm tra categoryid có tồn tại trong Category Tree
# --------------------------------------------------

category_tree_ids = set(
    pd.to_numeric(
        category_tree["categoryid"],
        errors="coerce"
    ).dropna()
)

property_category_ids = set(
    item_category_history["categoryid"]
    .dropna()
)

missing_in_tree = (
    property_category_ids
    - category_tree_ids
)

print("\n6. Category được sử dụng nhưng không có trong Category Tree:")
print(
    "Số category:",
    len(missing_in_tree)
)

if len(missing_in_tree) > 0:
    print(
        "Một số category:",
        sorted(missing_in_tree)[:20]
    )

# --------------------------------------------------
# 6. Kiểm tra category Tree có category không được sử dụng
# --------------------------------------------------

unused_tree_categories = (
    category_tree_ids
    - property_category_ids
)

print("\n7. Category trong Tree nhưng không xuất hiện trong Item Properties:")
print(
    "Số category:",
    len(unused_tree_categories)
)

# --------------------------------------------------
# 7. Ví dụ Item có nhiều Category
# --------------------------------------------------

print("\n8. Ví dụ Item có nhiều Category:")

example_items = (
    multiple_category_items
    .head(5)
    .index
)

example_category_history = (
    item_category_history[
        item_category_history["itemid"]
        .isin(example_items)
    ]
)

print(
    example_category_history[
        [
            "datetime",
            "itemid",
            "categoryid",
            "category_changed"
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# 8. Kiểm tra Category Tree
# --------------------------------------------------

print("\n9. Category Tree:")

print(
    "Tổng category:",
    category_tree["categoryid"].nunique()
)

print(
    "Missing parentid:",
    category_tree["parentid"].isna().sum()
)

===== ITEM–CATEGORY TEMPORAL CHECK =====


NameError: name 'item_properties_all' is not defined

In [ ]:
# BƯỚC 52 — Kiểm tra Category Tree có missing parentid

print("===== CATEGORY TREE ROOT CHECK =====")

category_tree_check = category_tree.copy()

print("\n1. Các Category có parentid bị thiếu:")

root_categories = category_tree_check[
    category_tree_check["parentid"].isna()
].copy()

print(
    root_categories[
        ["categoryid", "parentid"]
    ].to_string(index=False)
)

print("\n2. Số Category bị thiếu parentid:")
print(len(root_categories))

print("\n3. Kiểm tra categoryid có trùng không:")
print(
    "Duplicate categoryid:",
    category_tree_check["categoryid"].duplicated().sum()
)

print("\n4. Kiểm tra parentid có tồn tại trong categoryid không:")

category_ids = set(
    pd.to_numeric(
        category_tree_check["categoryid"],
        errors="coerce"
    ).dropna()
)

parent_ids = set(
    pd.to_numeric(
        category_tree_check["parentid"],
        errors="coerce"
    ).dropna()
)

parent_not_in_tree = parent_ids - category_ids

print(
    "Số parentid không tồn tại trong Category Tree:",
    len(parent_not_in_tree)
)

if len(parent_not_in_tree) > 0:
    print(
        "Một số parentid:",
        sorted(parent_not_in_tree)[:30]
    )

print("\n5. Category được sử dụng trong Item Properties có parentid thiếu:")

used_categories_with_missing_parent = root_categories[
    root_categories["categoryid"].isin(property_category_ids)
]

print(
    used_categories_with_missing_parent[
        ["categoryid", "parentid"]
    ].to_string(index=False)
)

print(
    "\nSố category root có xuất hiện trong Item Properties:",
    len(used_categories_with_missing_parent)
)

In [ ]:
# BƯỚC 53 — Kiểm tra Category không tồn tại trong Category Tree

print("===== CATEGORY ID CONSISTENCY CHECK =====")

# Danh sách 30 category được dùng trong Item Properties
missing_tree_categories = sorted(
    property_category_ids - category_tree_ids
)

print("\n1. Số Category không có trong Category Tree:")
print(len(missing_tree_categories))

print("\n2. Danh sách toàn bộ Category không có trong Tree:")
print(missing_tree_categories)

# Đếm số bản ghi Item Properties của từng category
missing_category_usage = (
    item_category_history[
        item_category_history["categoryid"].isin(
            missing_tree_categories
        )
    ]
    .groupby("categoryid")
    .agg(
        record_count=("categoryid", "size"),
        item_count=("itemid", "nunique")
    )
    .reset_index()
    .sort_values(
        by="record_count",
        ascending=False
    )
)

print("\n3. Mức độ sử dụng của các Category không có trong Tree:")
print(
    missing_category_usage.to_string(index=False)
)

print("\n4. Tổng số bản ghi sử dụng các Category này:")
print(
    missing_category_usage["record_count"].sum()
)

print("\n5. Tổng số Item sử dụng các Category này:")
print(
    missing_category_usage["item_count"].sum()
)

===== CATEGORY ID CONSISTENCY CHECK =====


NameError: name 'property_category_ids' is not defined

In [ ]:
import os

print("===== KIỂM TRA FILE DỮ LIỆU ĐÃ XUẤT =====")

files_to_check = [
    "/content/clean_events_final.csv",
    "/content/user_item_event_final.csv"
]

for file_path in files_to_check:
    print("\nFile:", file_path)

    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024**2)
        print("Tồn tại: YES")
        print("Dung lượng:", round(size_mb, 2), "MB")
    else:
        print("Tồn tại: NO")

===== KIỂM TRA FILE DỮ LIỆU ĐÃ XUẤT =====

File: /content/clean_events_final.csv
Tồn tại: NO

File: /content/user_item_event_final.csv
Tồn tại: NO


In [ ]:
import pandas as pd

clean_events_final = pd.read_csv(
    "/content/clean_events_final.csv",
    parse_dates=["datetime"]
)

print("Đã load clean_events_final")
print("Shape:", clean_events_final.shape)
print("Columns:", clean_events_final.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/content/clean_events_final.csv'

In [ ]:
# BƯỚC 54 — Kiểm tra độ phủ Item Properties so với Events

print("===== ITEM COVERAGE CHECK =====")

# Item xuất hiện trong dữ liệu hành vi
event_item_ids = set(
    clean_events_final["itemid"].dropna().unique()
)

# Item xuất hiện trong Item Properties
property_item_ids = set(
    item_properties_all["itemid"].dropna().unique()
)

print("\n1. Item trong Events:")
print(len(event_item_ids))

print("\n2. Item trong Item Properties:")
print(len(property_item_ids))

# Item có trong Events nhưng không có Properties
items_event_not_property = (
    event_item_ids - property_item_ids
)

print("\n3. Item có trong Events nhưng không có Item Properties:")
print(len(items_event_not_property))

print(
    "Tỷ lệ:",
    round(
        len(items_event_not_property)
        / len(event_item_ids)
        * 100,
        4
    ),
    "%"
)

# Item có Properties nhưng không xuất hiện trong Events
items_property_not_event = (
    property_item_ids - event_item_ids
)

print("\n4. Item có Item Properties nhưng không xuất hiện trong Events:")
print(len(items_property_not_event))

print(
    "Tỷ lệ:",
    round(
        len(items_property_not_event)
        / len(property_item_ids)
        * 100,
        4
    ),
    "%"
)

# Giao nhau
items_overlap = (
    event_item_ids & property_item_ids
)

print("\n5. Item xuất hiện ở cả hai nguồn:")
print(len(items_overlap))

print(
    "Tỷ lệ coverage Events:",
    round(
        len(items_overlap)
        / len(event_item_ids)
        * 100,
        4
    ),
    "%"
)

print(
    "Tỷ lệ coverage Item Properties:",
    round(
        len(items_overlap)
        / len(property_item_ids)
        * 100,
        4
    ),
    "%"
)

In [ ]:
# BƯỚC 55 — Kiểm tra Temporal Coverage giữa Events và Item Properties

print("===== TEMPORAL COVERAGE CHECK =====")

# Khoảng thời gian Events
event_time_min = clean_events_final["datetime"].min()
event_time_max = clean_events_final["datetime"].max()

# Khoảng thời gian Item Properties
property_time_min = item_properties_all["datetime"].min()
property_time_max = item_properties_all["datetime"].max()

print("\n1. Khoảng thời gian Events:")
print("Min:", event_time_min)
print("Max:", event_time_max)

print("\n2. Khoảng thời gian Item Properties:")
print("Min:", property_time_min)
print("Max:", property_time_max)

# Item xuất hiện trong Events nhưng không có Properties
event_items = (
    clean_events_final[
        clean_events_final["itemid"].isin(
            items_event_not_property
        )
    ]
)

print("\n3. Events của 49.815 item không có Properties:")

print(
    "Số event:",
    len(event_items)
)

print(
    "Số user:",
    event_items["visitorid"].nunique()
)

print(
    "Số session:",
    event_items["session_id"].nunique()
)

print(
    "Thời điểm event đầu tiên:",
    event_items["datetime"].min()
)

print(
    "Thời điểm event cuối cùng:",
    event_items["datetime"].max()
)

print("\n4. Phân bố loại Event của 49.815 item:")

print(
    event_items["event"]
    .value_counts()
)

print("\n5. Có transaction trong nhóm item không có Properties?")

print(
    "Transaction events:",
    (event_items["event"] == "transaction").sum()
)

print(
    "Transaction IDs:",
    event_items["transactionid"].nunique()
)

In [ ]:
# BƯỚC 56 — Kiểm tra Item thiếu Properties trong thời gian Property có dữ liệu

print("===== IN-WINDOW ITEM COVERAGE CHECK =====")

property_start = item_properties_all["datetime"].min()
property_end = item_properties_all["datetime"].max()

# Chỉ lấy Events nằm trong khoảng thời gian Item Properties có dữ liệu
events_in_property_window = clean_events_final[
    (clean_events_final["datetime"] >= property_start) &
    (clean_events_final["datetime"] <= property_end)
].copy()

print("\n1. Events nằm trong khoảng thời gian Item Properties:")
print(len(events_in_property_window))

print(
    "Thời gian:",
    events_in_property_window["datetime"].min(),
    "→",
    events_in_property_window["datetime"].max()
)

# Item trong Events trong khoảng thời gian này
event_items_in_window = set(
    events_in_property_window["itemid"]
    .dropna()
    .unique()
)

print("\n2. Item xuất hiện trong Events trong Property Window:")
print(len(event_items_in_window))

# Item không có Properties
missing_items_in_window = (
    event_items_in_window - property_item_ids
)

print("\n3. Item vẫn thiếu Properties trong Property Window:")
print(len(missing_items_in_window))

print(
    "Tỷ lệ:",
    round(
        len(missing_items_in_window)
        / len(event_items_in_window)
        * 100,
        4
    ),
    "%"
)

# Events thuộc nhóm item này
events_missing_in_window = events_in_property_window[
    events_in_property_window["itemid"].isin(
        missing_items_in_window
    )
]

print("\n4. Số Events của nhóm Item này:")
print(len(events_missing_in_window))

print("\n5. Phân bố Event:")
print(
    events_missing_in_window["event"]
    .value_counts()
)

print("\n6. Transaction Events:")
print(
    (events_missing_in_window["event"] == "transaction").sum()
)

print("\n7. Transaction IDs:")
print(
    events_missing_in_window["transactionid"].nunique()
)

===== IN-WINDOW ITEM COVERAGE CHECK =====


NameError: name 'item_properties_all' is not defined

In [ ]:
# BƯỚC 57 — Kiểm tra Temporal Availability của Item Properties

print("===== ITEM PROPERTY TEMPORAL AVAILABILITY =====")

# Lấy thời gian Property đầu tiên và cuối cùng của từng item
item_property_coverage = (
    item_properties_all
    .groupby("itemid")["datetime"]
    .agg(
        property_first_time="min",
        property_last_time="max",
        property_record_count="count"
    )
    .reset_index()
)

# Các item xuất hiện trong Events
event_item_time = (
    clean_events_final
    .groupby("itemid")["datetime"]
    .agg(
        event_first_time="min",
        event_last_time="max",
        event_count="count"
    )
    .reset_index()
)

# Ghép coverage của Property vào Event Item
item_temporal_coverage = event_item_time.merge(
    item_property_coverage,
    on="itemid",
    how="left"
)

# Xác định item không có property hoàn toàn
item_temporal_coverage["property_missing"] = (
    item_temporal_coverage["property_first_time"].isna()
)

# Với item có property, kiểm tra event có nằm trước/sau khoảng property
item_temporal_coverage["event_before_property"] = (
    item_temporal_coverage["event_first_time"]
    <
    item_temporal_coverage["property_first_time"]
)

item_temporal_coverage["event_after_property"] = (
    item_temporal_coverage["event_last_time"]
    >
    item_temporal_coverage["property_last_time"]
)

print("\n1. Tổng số Item trong Events:")
print(len(item_temporal_coverage))

print("\n2. Item hoàn toàn không có Property:")
print(
    item_temporal_coverage["property_missing"].sum()
)

print("\n3. Item có Property nhưng Event bắt đầu trước Property:")
print(
    (
        (~item_temporal_coverage["property_missing"]) &
        item_temporal_coverage["event_before_property"]
    ).sum()
)

print("\n4. Item có Property nhưng Event kết thúc sau Property:")
print(
    (
        (~item_temporal_coverage["property_missing"]) &
        item_temporal_coverage["event_after_property"]
    ).sum()
)

print("\n5. Item có Property bao phủ toàn bộ khoảng Event:")
print(
    (
        (~item_temporal_coverage["property_missing"]) &
        (~item_temporal_coverage["event_before_property"]) &
        (~item_temporal_coverage["event_after_property"])
    ).sum()
)

print("\n6. Thống kê số bản ghi Property / Item:")
print(
    item_temporal_coverage["property_record_count"]
    .describe()
)

print("\n7. Một số Item có Property nhưng thời gian không bao phủ Event:")

temporal_mismatch = item_temporal_coverage[
    (~item_temporal_coverage["property_missing"]) &
    (
        item_temporal_coverage["event_before_property"] |
        item_temporal_coverage["event_after_property"]
    )
]

print(
    temporal_mismatch[
        [
            "itemid",
            "event_first_time",
            "event_last_time",
            "property_first_time",
            "property_last_time",
            "property_record_count"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# BƯỚC 58 — Kiểm tra cấu trúc Item–Property

print("===== ITEM PROPERTY STRUCTURE CHECK =====")

# Số property khác nhau trên mỗi item
item_property_count = (
    item_properties_all
    .groupby("itemid")["property"]
    .nunique()
)

print("\n1. Phân phối số Property / Item:")
print(
    item_property_count.describe()
)

# Item có số property thấp / cao
print("\n2. Item có ít Property nhất:")
print(
    item_property_count.nsmallest(10)
)

print("\n3. Item có nhiều Property nhất:")
print(
    item_property_count.nlargest(10)
)

# Phân phối số item theo property
property_item_count = (
    item_properties_all
    .groupby("property")["itemid"]
    .nunique()
    .sort_values()
)

print("\n4. Phân phối số Item / Property:")
print(
    property_item_count.describe()
)

print("\n5. 10 Property có ít Item nhất:")
print(
    property_item_count.head(10)
)

print("\n6. 10 Property có nhiều Item nhất:")
print(
    property_item_count.tail(10)
)

# Property xuất hiện ở toàn bộ item
total_items_property = item_properties_all["itemid"].nunique()

properties_full_coverage = (
    property_item_count[
        property_item_count == total_items_property
    ]
)

print("\n7. Property xuất hiện ở 100% Item:")
print(
    len(properties_full_coverage)
)

print(
    "Tổng số Property:",
    item_properties_all["property"].nunique()
)

===== ITEM PROPERTY STRUCTURE CHECK =====


NameError: name 'item_properties_all' is not defined

In [ ]:

# BƯỚC 58A — Ghép hai file Item Properties

item_properties_all = pd.concat(
    [item_prop_1, item_prop_2],
    ignore_index=True
)

# Chuẩn hóa thời gian
item_properties_all["datetime"] = pd.to_datetime(
    item_properties_all["timestamp"],
    unit="ms"
)

# Sắp xếp theo item, property và thời gian
item_properties_all = item_properties_all.sort_values(
    by=["itemid", "property", "timestamp"]
).reset_index(drop=True)

print("===== KẾT QUẢ GHÉP ITEM PROPERTIES =====")
print("Shape:", item_properties_all.shape)
print("Số item:", item_properties_all["itemid"].nunique())
print("Số property:", item_properties_all["property"].nunique())
print("Missing:")
print(item_properties_all.isna().sum())

In [ ]:

# BƯỚC 59 — Kiểm tra kiểu dữ liệu và giá trị đặc biệt

import pandas as pd

print("===== ITEM PROPERTY VALUE CHECK =====")

# 1. Tạo lại value_changed
# Chỉ đánh dấu thay đổi khi value khác bản ghi trước
# của cùng itemid và property.
if "value_changed" not in item_properties_all.columns:
    previous_value = (
        item_properties_all
        .groupby(["itemid", "property"], sort=False)["value"]
        .shift(1)
    )

    item_properties_all["value_changed"] = (
        previous_value.notna()
        & item_properties_all["value"].ne(previous_value)
    ).astype("int64")

print("\n1. Kiểu dữ liệu:")
print(item_properties_all.dtypes)

print("\n2. Missing:")
print(
    item_properties_all[
        ["timestamp", "datetime", "itemid", "property", "value"]
    ].isna().sum()
)

print("\n3. Số giá trị rỗng trong value:")
print(
    item_properties_all["value"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

# 4. Ví dụ Value theo một số Property
sample_properties = [
    66, 926, 207, 8, 856,
    211, 197, 646, 823, 69,
    722, 213, 388, 233, 270,
    667, 867, 396, 325, 314
]

print("\n4. Một số Property và Value mẫu:")

for prop in sample_properties:
    values = (
        item_properties_all.loc[
            item_properties_all["property"].astype(str) == str(prop),
            "value"
        ]
        .drop_duplicates()
        .head(5)
        .tolist()
    )

    if values:
        print(f"Property = {prop} | Ví dụ Value = {values}")

# 5. Số Value khác nhau theo Property
value_count_by_property = (
    item_properties_all
    .groupby("property")["value"]
    .nunique()
)

print("\n5. Số Value khác nhau theo Property:")
print(value_count_by_property.describe())

# 6. Top 10 Property có nhiều Value khác nhau nhất
print("\n6. Property có nhiều Value khác nhau nhất:")
print(value_count_by_property.sort_values(ascending=False).head(10))

# 7. Kiểm tra tổng số lần value thay đổi
print("\n7. Tổng số lần value thay đổi:")
print(item_properties_all["value_changed"].sum())

print("\n===== KẾT THÚC ITEM PROPERTY VALUE CHECK =====")

===== ITEM PROPERTY VALUE CHECK =====


NameError: name 'item_properties_all' is not defined

In [ ]:

import pandas as pd
import os

# Nạp hai file Item Properties
item_prop_1 = pd.read_csv(
    "/content/data/item_properties_part1.csv"
)

item_prop_2 = pd.read_csv(
    "/content/data/item_properties_part2.csv"
)

# Nạp Category Tree
category_tree = pd.read_csv(
    "/content/data/category_tree.csv"
)

print("===== KIỂM TRA DỮ LIỆU ĐÃ NẠP =====")
print("item_prop_1:", item_prop_1.shape)
print("item_prop_2:", item_prop_2.shape)
print("category_tree:", category_tree.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/data/item_properties_part1.csv'

In [ ]:
import os

print("Các file trong thư mục /content/data:")
print(os.listdir("/content/data"))

In [ ]:
# BƯỚC 60 — Tạo Item Properties sạch

item_properties_final = (
    item_properties_all[
        [
            "timestamp",
            "datetime",
            "itemid",
            "property",
            "value"
        ]
    ]
    .sort_values(
        by=["itemid", "property", "timestamp"]
    )
    .reset_index(drop=True)
)

print("===== ITEM PROPERTIES FINAL =====")

print("\n1. Shape:")
print(item_properties_final.shape)

print("\n2. Columns:")
print(item_properties_final.columns.tolist())

print("\n3. Missing:")
print(
    item_properties_final.isna().sum()
)

print("\n4. Duplicate:")
print(
    item_properties_final.duplicated().sum()
)

print("\n5. Số Item:")
print(
    item_properties_final["itemid"].nunique()
)

print("\n6. Số Property:")
print(
    item_properties_final["property"].nunique()
)

print("\n7. Khoảng thời gian:")
print(
    item_properties_final["datetime"].min(),
    "→",
    item_properties_final["datetime"].max()
)

NameError: name 'item_properties_all' is not defined

In [ ]:
# BƯỚC 61 — Kiểm tra Category Tree trước khi tạo bản final

print("===== CATEGORY TREE CHECK =====")

print("\n1. Shape:")
print(category_tree.shape)

print("\n2. Columns:")
print(category_tree.columns.tolist())

print("\n3. Missing:")
print(
    category_tree.isna().sum()
)

print("\n4. Duplicate categoryid:")
print(
    category_tree["categoryid"].duplicated().sum()
)

print("\n5. Số Category:")
print(
    category_tree["categoryid"].nunique()
)

print("\n6. Số Parent Category khác nhau:")
print(
    category_tree["parentid"].nunique()
)

print("\n7. Các Category có parentid bị thiếu:")
print(
    category_tree[
        category_tree["parentid"].isna()
    ]["categoryid"].tolist()
)

print("\n8. Parentid không tồn tại trong Category Tree:")

invalid_parent = category_tree[
    category_tree["parentid"].notna()
    & ~category_tree["parentid"].isin(
        category_tree["categoryid"]
    )
]

print(invalid_parent)

print("\n9. Số parentid không hợp lệ:")
print(len(invalid_parent))

===== CATEGORY TREE CHECK =====

1. Shape:


NameError: name 'category_tree' is not defined

In [ ]:
# BƯỚC 62 — Tạo Category Tree Final

category_tree_final = (
    category_tree[
        [
            "categoryid",
            "parentid"
        ]
    ]
    .sort_values(
        by=["categoryid"]
    )
    .reset_index(drop=True)
)

print("===== CATEGORY TREE FINAL =====")

print("\n1. Shape:")
print(category_tree_final.shape)

print("\n2. Columns:")
print(category_tree_final.columns.tolist())

print("\n3. Missing:")
print(
    category_tree_final.isna().sum()
)

print("\n4. Duplicate:")
print(
    category_tree_final.duplicated().sum()
)

print("\n5. Duplicate categoryid:")
print(
    category_tree_final["categoryid"].duplicated().sum()
)

print("\n6. Số Category:")
print(
    category_tree_final["categoryid"].nunique()
)

print("\n7. Số Root Category:")
print(
    category_tree_final["parentid"].isna().sum()
)

print("\n8. Parentid không tồn tại:")

invalid_parent = category_tree_final[
    category_tree_final["parentid"].notna()
    & ~category_tree_final["parentid"].isin(
        category_tree_final["categoryid"]
    )
]

print(len(invalid_parent))

NameError: name 'category_tree' is not defined

In [ ]:
# BƯỚC 63 — Tạo bảng lịch sử Item–Category

item_category_history = (
    item_properties_final[
        item_properties_final["property"] == "categoryid"
    ][
        [
            "timestamp",
            "datetime",
            "itemid",
            "value"
        ]
    ]
    .rename(
        columns={
            "value": "categoryid"
        }
    )
    .copy()
)

# Chuẩn hóa categoryid về số nguyên
item_category_history["categoryid"] = pd.to_numeric(
    item_category_history["categoryid"],
    errors="coerce"
)

item_category_history = (
    item_category_history
    .dropna(subset=["categoryid"])
    .copy()
)

item_category_history["categoryid"] = (
    item_category_history["categoryid"]
    .astype("int64")
)

item_category_history = (
    item_category_history
    .sort_values(
        by=["itemid", "timestamp"]
    )
    .reset_index(drop=True)
)

print("===== ITEM CATEGORY HISTORY =====")

print("\n1. Shape:")
print(item_category_history.shape)

print("\n2. Columns:")
print(item_category_history.columns.tolist())

print("\n3. Missing:")
print(
    item_category_history.isna().sum()
)

print("\n4. Duplicate:")
print(
    item_category_history.duplicated().sum()
)

print("\n5. Số Item:")
print(
    item_category_history["itemid"].nunique()
)

print("\n6. Số Category:")
print(
    item_category_history["categoryid"].nunique()
)

print("\n7. Khoảng thời gian:")
print(
    item_category_history["datetime"].min(),
    "→",
    item_category_history["datetime"].max()
)

print("\n8. Category không có trong Category Tree:")

missing_categories = sorted(
    set(item_category_history["categoryid"].unique())
    - set(category_tree_final["categoryid"].unique())
)

print(missing_categories)

print("\n9. Số Category không có trong Tree:")
print(len(missing_categories))

In [ ]:
# BƯỚC 64 — Kiểm tra thay đổi Category theo thời gian

print("===== CATEGORY TEMPORAL CHECK =====")

# 1. Số category khác nhau trên mỗi item
category_per_item = (
    item_category_history
    .groupby("itemid")["categoryid"]
    .nunique()
)

print("\n1. Phân bố số Category khác nhau / Item:")
print(
    category_per_item.describe()
)

print("\n2. Item có nhiều hơn 1 Category:")
print(
    (category_per_item > 1).sum()
)

print(
    "Tỷ lệ:",
    round(
        (category_per_item > 1).mean() * 100,
        2
    ),
    "%"
)

# 2. Số lần thay đổi Category
item_category_history["category_changed"] = (
    item_category_history
    .groupby("itemid")["categoryid"]
    .diff()
    .fillna(0)
    .ne(0)
)

total_changes = (
    item_category_history["category_changed"]
    .sum()
)

print("\n3. Tổng số lần thay đổi Category:")
print(total_changes)

# 3. Kiểm tra cùng item + timestamp có nhiều category
same_item_timestamp = (
    item_category_history
    .groupby(["itemid", "timestamp"])["categoryid"]
    .nunique()
)

conflicting_timestamp = (
    same_item_timestamp > 1
).sum()

print(
    "\n4. Item có nhiều Category tại cùng một timestamp:"
)
print(conflicting_timestamp)

# 4. Kiểm tra timestamp giảm trong cùng item
time_diff = (
    item_category_history
    .groupby("itemid")["timestamp"]
    .diff()
)

print(
    "\n5. Timestamp giảm trong cùng Item:"
)
print(
    (time_diff < 0).sum()
)

# 5. Một số ví dụ item thay đổi category
changed_items = category_per_item[
    category_per_item > 1
].index

print(
    "\n6. Một số Item có thay đổi Category:"
)

for item in changed_items[:10]:
    print(
        "\nItem:", item
    )
    print(
        item_category_history[
            item_category_history["itemid"] == item
        ][
            [
                "datetime",
                "categoryid"
            ]
        ].drop_duplicates().to_string(index=False)
    )

===== CATEGORY TEMPORAL CHECK =====


NameError: name 'item_category_history' is not defined

In [ ]:
# BƯỚC 65 — Tạo Item Category History Final

item_category_history_final = (
    item_category_history[
        [
            "timestamp",
            "datetime",
            "itemid",
            "categoryid"
        ]
    ]
    .sort_values(
        by=["itemid", "timestamp"]
    )
    .reset_index(drop=True)
)

print("===== ITEM CATEGORY HISTORY FINAL =====")

print("\n1. Shape:")
print(item_category_history_final.shape)

print("\n2. Columns:")
print(item_category_history_final.columns.tolist())

print("\n3. Missing:")
print(
    item_category_history_final.isna().sum()
)

print("\n4. Duplicate:")
print(
    item_category_history_final.duplicated().sum()
)

print("\n5. Số Item:")
print(
    item_category_history_final["itemid"].nunique()
)

print("\n6. Số Category:")
print(
    item_category_history_final["categoryid"].nunique()
)

print("\n7. Khoảng thời gian:")
print(
    item_category_history_final["datetime"].min(),
    "→",
    item_category_history_final["datetime"].max()
)

print("\n8. Số Category không có trong Category Tree:")

missing_categories = (
    set(item_category_history_final["categoryid"].unique())
    - set(category_tree_final["categoryid"].unique())
)

print(len(missing_categories))

NameError: name 'item_category_history' is not defined

In [ ]:
# BƯỚC 66 — Ghép Item Category với Category Tree

item_category_with_parent = (
    item_category_history_final
    .merge(
        category_tree_final,
        on="categoryid",
        how="left",
        validate="many_to_one"
    )
)

print("===== ITEM CATEGORY WITH PARENT =====")

print("\n1. Shape:")
print(item_category_with_parent.shape)

print("\n2. Columns:")
print(item_category_with_parent.columns.tolist())

print("\n3. Missing:")
print(
    item_category_with_parent.isna().sum()
)

print("\n4. Duplicate:")
print(
    item_category_with_parent.duplicated().sum()
)

print("\n5. Số Item:")
print(
    item_category_with_parent["itemid"].nunique()
)

print("\n6. Số Category:")
print(
    item_category_with_parent["categoryid"].nunique()
)

print("\n7. Số Parent Category:")
print(
    item_category_with_parent["parentid"].nunique()
)

print("\n8. Số bản ghi không tìm thấy Parent:")

print(
    item_category_with_parent["parentid"].isna().sum()
)

print("\n9. Số bản ghi thuộc Root Category:")

print(
    item_category_with_parent["categoryid"]
    .isin(
        category_tree_final.loc[
            category_tree_final["parentid"].isna(),
            "categoryid"
        ]
    )
    .sum()
)

NameError: name 'item_category_history_final' is not defined

In [ ]:
# BƯỚC 67 — Phân loại Missing Parent

tree_categories = set(
    category_tree_final["categoryid"]
)

root_categories = set(
    category_tree_final.loc[
        category_tree_final["parentid"].isna(),
        "categoryid"
    ]
)

missing_tree_categories = set(
    item_category_with_parent.loc[
        item_category_with_parent["parentid"].isna(),
        "categoryid"
    ]
) - root_categories

print("===== MISSING PARENT ANALYSIS =====")

print("\n1. Tổng bản ghi thiếu parentid:")
print(
    item_category_with_parent["parentid"].isna().sum()
)

print("\n2. Bản ghi thuộc Root Category:")
print(
    item_category_with_parent[
        item_category_with_parent["categoryid"].isin(
            root_categories
        )
    ].shape[0]
)

print("\n3. Bản ghi thuộc Category không có trong Tree:")
print(
    item_category_with_parent[
        item_category_with_parent["categoryid"].isin(
            missing_tree_categories
        )
    ].shape[0]
)

print("\n4. Số Category không có trong Tree:")
print(
    len(missing_tree_categories)
)

print("\n5. Danh sách Category không có trong Tree:")
print(
    sorted(missing_tree_categories)
)

print("\n6. Kiểm tra tổng:")
print(
    51 + 167,
    "=?",
    item_category_with_parent["parentid"].isna().sum()
)

NameError: name 'category_tree_final' is not defined

4 file sau BƯỚC 68

File	Nội dung	Vai trò
item_properties_final.csv	Lịch sử toàn bộ Property	Bảng Item Properties chính
item_category_history_final.csv	Lịch sử Category của Item	Bảng Item–Category theo thời gian
category_tree_final.csv	Category → Parent	Cây Category gốc
item_category_with_parent.csv	Item → Category → Parent	Bảng Category đã bổ sung Parent

Lưu ý: item_category_with_parent.csv là bảng hỗ trợ phân tích; không thay thế item_category_history_final.csv.

In [ ]:
# BƯỚC 68 — Xuất các file sạch của Phase E

# 1. Item Properties
item_properties_final.to_csv(
    "/content/item_properties_final.csv",
    index=False
)

# 2. Item Category History
item_category_history_final.to_csv(
    "/content/item_category_history_final.csv",
    index=False
)

# 3. Category Tree
category_tree_final.to_csv(
    "/content/category_tree_final.csv",
    index=False
)

# 4. Item Category with Parent
item_category_with_parent.to_csv(
    "/content/item_category_with_parent.csv",
    index=False
)

print("===== EXPORT PHASE E =====")

print("\nĐã xuất thành công 4 file:")

print("1. /content/item_properties_final.csv")
print("2. /content/item_category_history_final.csv")
print("3. /content/category_tree_final.csv")
print("4. /content/item_category_with_parent.csv")

NameError: name 'item_properties_final' is not defined

In [ ]:
# BƯỚC 69 — Kiểm tra dữ liệu đầu vào cho Phase F

print("===== PHASE F DATA CHECK =====")

required_objects = [
    "clean_events_final",
    "user_item_event_final",
    "item_properties_final",
    "item_category_history_final",
    "category_tree_final",
    "item_category_with_parent"
]

for obj in required_objects:
    print(
        f"{obj}:",
        "OK" if obj in globals() else "MISSING"
    )

===== PHASE F DATA CHECK =====
clean_events_final: MISSING
user_item_event_final: MISSING
item_properties_final: MISSING
item_category_history_final: MISSING
category_tree_final: MISSING
item_category_with_parent: MISSING


In [ ]:
# BƯỚC 70 — Reload User–Item–Event Final

import pandas as pd

user_item_event_final = pd.read_csv(
    "/content/user_item_event_final.csv",
    parse_dates=["datetime"]
)

print("===== USER ITEM EVENT FINAL =====")

print("\n1. Shape:")
print(user_item_event_final.shape)

print("\n2. Columns:")
print(user_item_event_final.columns.tolist())

print("\n3. Missing:")
print(
    user_item_event_final.isna().sum()
)

print("\n4. Duplicate:")
print(
    user_item_event_final.duplicated().sum()
)

print("\n5. Số User:")
print(
    user_item_event_final["visitorid"].nunique()
)

print("\n6. Số Item:")
print(
    user_item_event_final["itemid"].nunique()
)

print("\n7. Số Session:")
print(
    user_item_event_final["session_id"].nunique()
)

print("\n8. Số Transaction:")
print(
    user_item_event_final["transactionid"].nunique()
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/user_item_event_final.csv'

In [ ]:

import pandas as pd
import os

file_path = "/content/clean_events_final.csv"

if os.path.exists(file_path):
    clean_events_final = pd.read_csv(
        file_path,
        parse_dates=["datetime"]
    )

    print("Đã khôi phục clean_events_final!")
    print("Shape:", clean_events_final.shape)
    print("Columns:", clean_events_final.columns.tolist())
    print("\nMissing:")
    print(clean_events_final.isna().sum())
else:
    print("Không tìm thấy file:", file_path)

In [ ]:
# BƯỚC 71 — EDA F1: Tổng quan Event

print("===== EDA F1 — EVENT OVERVIEW =====")

# 1. Tổng số Event
print("\n1. Tổng số Event:")
print(
    len(clean_events_final)
)

# 2. Phân bố Event
event_count = (
    clean_events_final["event"]
    .value_counts()
)

event_pct = (
    clean_events_final["event"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

event_summary = pd.DataFrame({
    "count": event_count,
    "percentage": event_pct
})

print("\n2. Phân bố Event:")
print(event_summary)

# 3. Số User theo từng loại Event
user_by_event = (
    clean_events_final
    .groupby("event")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n3. Số User theo Event:")
print(user_by_event)

# 4. Số Item theo từng loại Event
item_by_event = (
    clean_events_final
    .groupby("event")["itemid"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n4. Số Item theo Event:")
print(item_by_event)

# 5. Số Session theo từng loại Event
session_by_event = (
    clean_events_final
    .groupby("event")["session_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n5. Số Session theo Event:")
print(session_by_event)

# 6. Khoảng thời gian dữ liệu
print("\n6. Khoảng thời gian:")
print(
    clean_events_final["datetime"].min(),
    "→",
    clean_events_final["datetime"].max()
)

# 7. Tổng số ngày quan sát
observation_days = (
    clean_events_final["datetime"].max()
    - clean_events_final["datetime"].min()
).total_seconds() / 86400

print("\n7. Số ngày quan sát:")
print(round(observation_days, 2))

===== EDA F1 — EVENT OVERVIEW =====

1. Tổng số Event:


NameError: name 'clean_events_final' is not defined

In [ ]:
# BƯỚC 72 — EDA F2: HÀNH VI THEO THỜI GIAN

print("===== EDA F2 — TIME BEHAVIOR =====")

# Tạo các biến thời gian tạm thời
eda_time = clean_events_final[
    ["datetime", "event"]
].copy()

eda_time["date"] = eda_time["datetime"].dt.date
eda_time["hour"] = eda_time["datetime"].dt.hour
eda_time["day_of_week"] = eda_time["datetime"].dt.day_name()
eda_time["month"] = eda_time["datetime"].dt.to_period("M").astype(str)

# 1. Số Event theo ngày
daily_event = (
    eda_time
    .groupby("date")
    .size()
)

print("\n1. Event theo ngày:")
print(daily_event.describe())

print("\n5 ngày có nhiều Event nhất:")
print(
    daily_event
    .sort_values(ascending=False)
    .head(5)
)

print("\n5 ngày có ít Event nhất:")
print(
    daily_event
    .sort_values()
    .head(5)
)

# 2. Event theo giờ
hourly_event = (
    eda_time
    .groupby("hour")
    .size()
)

print("\n2. Event theo giờ:")
print(hourly_event)

# 3. Event theo thứ trong tuần
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_event = (
    eda_time
    .groupby("day_of_week")
    .size()
    .reindex(weekday_order)
)

print("\n3. Event theo thứ trong tuần:")
print(weekday_event)

# 4. Event theo tháng
monthly_event = (
    eda_time
    .groupby("month")
    .size()
)

print("\n4. Event theo tháng:")
print(monthly_event)

# 5. Event theo ngày và loại Event
daily_event_type = pd.crosstab(
    eda_time["date"],
    eda_time["event"]
)

print("\n5. Event theo ngày và loại Event:")
print(daily_event_type.head())

# 6. Transaction theo ngày
daily_transaction = (
    eda_time[
        eda_time["event"] == "transaction"
    ]
    .groupby("date")
    .size()
)

print("\n6. Transaction theo ngày:")
print(daily_transaction.describe())

print("\n5 ngày có nhiều Transaction nhất:")
print(
    daily_transaction
    .sort_values(ascending=False)
    .head(5)
)

# 7. Kiểm tra ngày có Event
print("\n7. Tổng số ngày có Event:")
print(
    eda_time["date"].nunique()
)

Kết luận BƯỚC 72
Theo tháng: tháng 7 có nhiều Event nhất (697,849), tháng 9 thấp nhất (303,655), một phần do dữ liệu kết thúc ngày 18/09.
Theo giờ: hoạt động tăng mạnh từ khoảng 15h, cao nhất vào 20h (187,887 Event), sau đó giảm dần.
Theo thứ: Thứ Ba có nhiều Event nhất (447,006), Thứ Bảy thấp nhất (305,167).
Transaction: trung bình khoảng 162 giao dịch/ngày, cao nhất 276 giao dịch vào 16/06/2015.
Theo ngày: có biến động đáng kể; ngày 26/07 có 32,692 Event, trong khi 18/09 chỉ có 1,528 Event vì đây là ngày cuối kỳ dữ liệu.

👉 Chưa cần xử lý hay loại bỏ các ngày/giờ có giá trị thấp hoặc cao, vì đây là biến động quan sát thực tế của dữ liệu.

In [ ]:
# BƯỚC 73 — EDA F3: USER BEHAVIOR

print("===== EDA F3 — USER BEHAVIOR =====")

# 1. Số Event trên mỗi User
events_per_user = (
    user_item_event_final
    .groupby("visitorid")
    .size()
)

print("\n1. Event trên mỗi User:")
print(events_per_user.describe())

# 2. Số Session trên mỗi User
sessions_per_user = (
    user_item_event_final
    .groupby("visitorid")["session_id"]
    .nunique()
)

print("\n2. Session trên mỗi User:")
print(sessions_per_user.describe())

# 3. Số Item khác nhau trên mỗi User
items_per_user = (
    user_item_event_final
    .groupby("visitorid")["itemid"]
    .nunique()
)

print("\n3. Item khác nhau trên mỗi User:")
print(items_per_user.describe())

# 4. Số User theo số lượng Event
user_event_distribution = (
    events_per_user
    .value_counts()
    .sort_index()
)

print("\n4. Phân bố User theo số Event:")
print(user_event_distribution.head(20))

# 5. User có nhiều Event nhất
print("\n5. Top 10 User có nhiều Event nhất:")
print(
    events_per_user
    .sort_values(ascending=False)
    .head(10)
)

# 6. User có Transaction
users_with_transaction = (
    user_item_event_final[
        user_item_event_final["event"] == "transaction"
    ]["visitorid"]
    .nunique()
)

total_users = user_item_event_final["visitorid"].nunique()

print("\n6. User có Transaction:")
print(users_with_transaction)

print("Tỷ lệ User có Transaction:")
print(
    round(
        users_with_transaction / total_users * 100,
        2
    ),
    "%"
)

# 7. User có Add-to-cart
users_with_addtocart = (
    user_item_event_final[
        user_item_event_final["event"] == "addtocart"
    ]["visitorid"]
    .nunique()
)

print("\n7. User có Add-to-cart:")
print(users_with_addtocart)

print("Tỷ lệ User có Add-to-cart:")
print(
    round(
        users_with_addtocart / total_users * 100,
        2
    ),
    "%"
)

# 8. User chỉ View
user_event_types = (
    user_item_event_final
    .groupby("visitorid")["event"]
    .agg(lambda x: set(x))
)

users_view_only = (
    user_event_types
    .apply(lambda x: x == {"view"})
    .sum()
)

print("\n8. User chỉ có View:")
print(users_view_only)

print("Tỷ lệ User chỉ View:")
print(
    round(
        users_view_only / total_users * 100,
        2
    ),
    "%"
)

===== EDA F3 — USER BEHAVIOR =====


NameError: name 'user_item_event_final' is not defined

Nhận xét quan trọng:

Phần lớn User có rất ít hoạt động: median chỉ 1 Event/User.
97.24% User chỉ phát sinh View, cho thấy hành vi mua hàng là rất hiếm.
Chỉ 0.83% User có Transaction, xác nhận dữ liệu có mức mất cân bằng lớp rất cao đối với bài toán dự đoán mua hàng.
Một số User có số Event rất lớn, như User 1150086 có 7,757 Event. Đây là hành vi cực trị (heavy users) nhưng chưa có lý do để loại bỏ.
Vì vậy, giữ nguyên toàn bộ User, không lọc User ít Event và không cắt các User có nhiều Event ở bước EDA.

In [ ]:
# BƯỚC 74 — EDA F4: ITEM BEHAVIOR

print("===== EDA F4 — ITEM BEHAVIOR =====")

# 1. Số Event trên mỗi Item
events_per_item = (
    user_item_event_final
    .groupby("itemid")
    .size()
)

print("\n1. Event trên mỗi Item:")
print(events_per_item.describe())

# 2. Số User khác nhau trên mỗi Item
users_per_item = (
    user_item_event_final
    .groupby("itemid")["visitorid"]
    .nunique()
)

print("\n2. User khác nhau trên mỗi Item:")
print(users_per_item.describe())

# 3. Số Session khác nhau trên mỗi Item
sessions_per_item = (
    user_item_event_final
    .groupby("itemid")["session_id"]
    .nunique()
)

print("\n3. Session khác nhau trên mỗi Item:")
print(sessions_per_item.describe())

# 4. Số Event theo Item
item_event_distribution = (
    events_per_item
    .value_counts()
    .sort_index()
)

print("\n4. Phân bố Item theo số Event:")
print(item_event_distribution.head(20))

# 5. Top 10 Item có nhiều Event nhất
print("\n5. Top 10 Item có nhiều Event nhất:")
print(
    events_per_item
    .sort_values(ascending=False)
    .head(10)
)

# 6. Top 10 Item có nhiều User nhất
print("\n6. Top 10 Item có nhiều User nhất:")
print(
    users_per_item
    .sort_values(ascending=False)
    .head(10)
)

# 7. Item có Transaction
items_with_transaction = (
    user_item_event_final[
        user_item_event_final["event"] == "transaction"
    ]["itemid"]
    .nunique()
)

total_items = user_item_event_final["itemid"].nunique()

print("\n7. Item có Transaction:")
print(items_with_transaction)

print("Tỷ lệ Item có Transaction:")
print(
    round(
        items_with_transaction / total_items * 100,
        2
    ),
    "%"
)

# 8. Item chỉ có View
item_event_types = (
    user_item_event_final
    .groupby("itemid")["event"]
    .agg(lambda x: set(x))
)

items_view_only = (
    item_event_types
    .apply(lambda x: x == {"view"})
    .sum()
)

print("\n8. Item chỉ có View:")
print(items_view_only)

print("Tỷ lệ Item chỉ View:")
print(
    round(
        items_view_only / total_items * 100,
        2
    ),
    "%"
)

Kết luận BƯỚC 74

Có 235,061 Item, nhưng mức độ tương tác rất không đồng đều.
Median chỉ 3 Event/Item, trong khi Item nhiều nhất có 3,412 Event → phân phối lệch phải mạnh.
Median chỉ 2 User/Item, cho thấy nhiều sản phẩm có rất ít người tương tác.
89.66% Item chỉ có View, trong khi chỉ 5.12% Item có Transaction.
Một số Item có lượng tương tác rất lớn, nhưng đây chưa phải lý do để loại bỏ → giữ nguyên Item.
Kết quả này cũng củng cố rằng dữ liệu có cấu trúc long-tail/sparse, cần lưu ý khi xây dựng các mô hình ML sau này.

In [ ]:
# BƯỚC 75 — EDA F5: SESSION BEHAVIOR

print("===== EDA F5 — SESSION BEHAVIOR =====")

# 1. Số Event trên mỗi Session
events_per_session = (
    user_item_event_final
    .groupby("session_id")
    .size()
)

print("\n1. Event trên mỗi Session:")
print(events_per_session.describe())

# 2. Số Item khác nhau trên mỗi Session
items_per_session = (
    user_item_event_final
    .groupby("session_id")["itemid"]
    .nunique()
)

print("\n2. Item khác nhau trên mỗi Session:")
print(items_per_session.describe())

# 3. Số Event theo loại trong mỗi Session
session_event_types = (
    user_item_event_final
    .groupby(["session_id", "event"])
    .size()
    .unstack(fill_value=0)
)

print("\n3. Phân bố loại Event trong Session:")
print(session_event_types.describe())

# 4. Session chỉ View
sessions_view_only = (
    session_event_types
    .fillna(0)
    .assign(
        view_only=lambda x:
        (x.get("view", 0) > 0)
        & (x.get("addtocart", 0) == 0)
        & (x.get("transaction", 0) == 0)
    )["view_only"]
    .sum()
)

total_sessions = user_item_event_final["session_id"].nunique()

print("\n4. Session chỉ View:")
print(sessions_view_only)

print("Tỷ lệ Session chỉ View:")
print(
    round(
        sessions_view_only / total_sessions * 100,
        2
    ),
    "%"
)

# 5. Session có Add-to-cart
sessions_with_addtocart = (
    user_item_event_final[
        user_item_event_final["event"] == "addtocart"
    ]["session_id"]
    .nunique()
)

print("\n5. Session có Add-to-cart:")
print(sessions_with_addtocart)

print("Tỷ lệ Session có Add-to-cart:")
print(
    round(
        sessions_with_addtocart / total_sessions * 100,
        2
    ),
    "%"
)

# 6. Session có Transaction
sessions_with_transaction = (
    user_item_event_final[
        user_item_event_final["event"] == "transaction"
    ]["session_id"]
    .nunique()
)

print("\n6. Session có Transaction:")
print(sessions_with_transaction)

print("Tỷ lệ Session có Transaction:")
print(
    round(
        sessions_with_transaction / total_sessions * 100,
        2
    ),
    "%"
)

# 7. Top 10 Session có nhiều Event nhất
print("\n7. Top 10 Session có nhiều Event nhất:")
print(
    events_per_session
    .sort_values(ascending=False)
    .head(10)
)

# 8. Session có nhiều Item nhất
print("\n8. Top 10 Session có nhiều Item khác nhau nhất:")
print(
    items_per_session
    .sort_values(ascending=False)
    .head(10)
)

Kết luận BƯỚC 75

Có 1,761,674 Session, trung bình 1.56 Event/Session.
Median chỉ 1 Event/Session, cho thấy phần lớn Session rất ngắn.
97.37% Session chỉ có View.
2.49% Session có Add-to-cart.
Chỉ 0.81% Session có Transaction.
Session nhiều Event nhất có 417 Event, còn Session nhiều Item nhất có 389 Item khác nhau.
Phân phối Session rất lệch phải, nhưng đây là đặc điểm hành vi thực tế → không loại bỏ Session dài.
Kết quả tiếp tục xác nhận dữ liệu có tính sparse + long-tail + purchase imbalance.



In [ ]:
# BƯỚC 76 — EDA F6: TRANSACTION & PURCHASE BEHAVIOR

print("===== EDA F6 — TRANSACTION & PURCHASE BEHAVIOR =====")

# Chỉ lấy các event transaction
transactions = user_item_event_final[
    user_item_event_final["event"] == "transaction"
].copy()

# 1. Tổng Transaction Event
print("\n1. Tổng Transaction Event:")
print(len(transactions))

# 2. Transaction ID duy nhất
unique_transactions = (
    transactions["transactionid"]
    .nunique()
)

print("\n2. Transaction ID duy nhất:")
print(unique_transactions)

# 3. Transaction theo User
transactions_per_user = (
    transactions
    .groupby("visitorid")["transactionid"]
    .nunique()
)

print("\n3. Transaction/User:")
print(transactions_per_user.describe())

# 4. Transaction theo Session
transactions_per_session = (
    transactions
    .groupby("session_id")["transactionid"]
    .nunique()
)

print("\n4. Transaction/Session:")
print(transactions_per_session.describe())

# 5. Số Item trong mỗi Transaction ID
items_per_transaction = (
    transactions
    .groupby("transactionid")["itemid"]
    .count()
)

print("\n5. Item/Transaction:")
print(items_per_transaction.describe())

# 6. Số Item khác nhau trong mỗi Transaction
unique_items_per_transaction = (
    transactions
    .groupby("transactionid")["itemid"]
    .nunique()
)

print("\n6. Item khác nhau/Transaction:")
print(unique_items_per_transaction.describe())

# 7. Transaction có nhiều Item nhất
print("\n7. Top 10 Transaction có nhiều Item nhất:")
print(
    items_per_transaction
    .sort_values(ascending=False)
    .head(10)
)

# 8. User có nhiều Transaction nhất
print("\n8. Top 10 User có nhiều Transaction nhất:")
print(
    transactions_per_user
    .sort_values(ascending=False)
    .head(10)
)

# 9. Item được mua nhiều nhất
items_purchased = (
    transactions
    .groupby("itemid")
    .size()
)

print("\n9. Top 10 Item được mua nhiều nhất:")
print(
    items_purchased
    .sort_values(ascending=False)
    .head(10)
)

# 10. Transaction ID có nhiều hơn 1 Item
multi_item_transactions = (
    (items_per_transaction > 1)
    .sum()
)

print("\n10. Transaction có >1 Item:")
print(multi_item_transactions)

print("Tỷ lệ Transaction có >1 Item:")
print(
    round(
        multi_item_transactions / unique_transactions * 100,
        2
    ),
    "%"
)

# 11. Transaction có Item trùng lặp
duplicate_item_transaction = (
    items_per_transaction
    > unique_items_per_transaction
)

print("\n11. Transaction có Item bị lặp:")
print(
    duplicate_item_transaction.sum()
)

print("Tỷ lệ Transaction có Item bị lặp:")
print(
    round(
        duplicate_item_transaction.sum()
        / unique_transactions * 100,
        2
    ),
    "%"
)

Điểm đáng chú ý:

Phần lớn người mua chỉ có 1 Transaction, nhưng một số User có tần suất mua rất cao.
Transaction 7063 có tới 31 Item.
Chỉ 0.46% Transaction có Item lặp, phù hợp với quyết định trước đó là không tự ý loại các dòng trùng item trong cùng transaction, vì dataset không có biến số lượng.
Item 461686 là sản phẩm được mua nhiều nhất với 133 lần.
Phân phối mua hàng có long-tail rất mạnh, vì vậy không nên dùng trung bình đơn thuần để mô tả hành vi mua.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# BƯỚC 77 — EDA F7: MULTI-BEHAVIOR & EVENT SEQUENCE

print("===== EDA F7 — MULTI-BEHAVIOR & EVENT SEQUENCE =====")

# 1. Tổ hợp Event theo User-Item
user_item_event_types = (
    user_item_event_final
    .groupby(["visitorid", "itemid"])["event"]
    .agg(lambda x: tuple(sorted(set(x))))
)

user_item_event_combo = (
    user_item_event_types
    .value_counts()
)

print("\n1. Tổ hợp Event theo User-Item:")
print(user_item_event_combo)

# 2. Số User-Item có nhiều loại Event
multi_behavior_user_item = (
    user_item_event_types
    .apply(lambda x: len(x) > 1)
    .sum()
)

total_user_item = (
    user_item_event_types.shape[0]
)

print("\n2. User-Item có nhiều hơn 1 loại Event:")
print(multi_behavior_user_item)

print("Tỷ lệ:")
print(
    round(
        multi_behavior_user_item
        / total_user_item * 100,
        2
    ),
    "%"
)

# 3. Tổ hợp Event theo Session
session_event_types = (
    user_item_event_final
    .groupby("session_id")["event"]
    .agg(lambda x: tuple(sorted(set(x))))
)

session_event_combo = (
    session_event_types
    .value_counts()
)

print("\n3. Tổ hợp Event theo Session:")
print(session_event_combo)

# 4. Session có nhiều loại Event
multi_behavior_session = (
    session_event_types
    .apply(lambda x: len(x) > 1)
    .sum()
)

total_session = session_event_types.shape[0]

print("\n4. Session có nhiều hơn 1 loại Event:")
print(multi_behavior_session)

print("Tỷ lệ:")
print(
    round(
        multi_behavior_session
        / total_session * 100,
        2
    ),
    "%"
)

# 5. Kiểm tra thứ tự Event trong Session
session_sequences = (
    user_item_event_final
    .sort_values(
        ["session_id", "datetime"]
    )
    .groupby("session_id")["event"]
    .agg(tuple)
)

sequence_counts = (
    session_sequences
    .value_counts()
)

print("\n5. Top 20 chuỗi Event trong Session:")
print(
    sequence_counts.head(20)
)

# 6. Transaction có phải Event cuối Session không
transaction_sessions = (
    user_item_event_final[
        user_item_event_final["event"] == "transaction"
    ]["session_id"]
    .unique()
)

session_last_event = (
    user_item_event_final
    .sort_values(
        ["session_id", "datetime"]
    )
    .groupby("session_id")["event"]
    .last()
)

transaction_last = (
    session_last_event.loc[transaction_sessions]
    == "transaction"
)

print("\n6. Transaction là Event cuối Session:")
print(
    transaction_last.sum()
)

print("Transaction không phải Event cuối Session:")
print(
    (~transaction_last).sum()
)

print("Tỷ lệ Transaction là Event cuối Session:")
print(
    round(
        transaction_last.mean() * 100,
        2
    ),
    "%"
)

===== EDA F7 — MULTI-BEHAVIOR & EVENT SEQUENCE =====


NameError: name 'user_item_event_final' is not defined

Kết luận BƯỚC 77

97.53% User–Item chỉ có một loại hành vi; 2.47% có nhiều loại hành vi.
Có 52,890 User–Item xuất hiện nhiều hành vi như view → addtocart → transaction.
Có 41,184 Session (2.34%) chứa nhiều loại Event.
Chuỗi phổ biến nhất vẫn là View đơn lẻ, nhưng tồn tại nhiều chuỗi tương tác phức tạp.
Chỉ 69.90% Transaction là Event cuối Session; có 4,303 Session (30.10%) tiếp tục có Event sau Transaction.

👉 Vì vậy, không được giả định Transaction luôn là điểm kết thúc Session và cũng không được ép toàn bộ dữ liệu thành funnel tuyến tính. Quyết định trước đó là giữ nguyên Multi-Behavior là đúng.

In [ ]:
# BƯỚC 78 — EDA F8: ITEM PROPERTIES & CATEGORY COVERAGE

print("===== EDA F8 — ITEM PROPERTIES & CATEGORY COVERAGE =====")

# Tập Item trong Event
event_items = set(
    user_item_event_final["itemid"].unique()
)

# Tập Item có Properties
property_items = set(
    item_properties_final["itemid"].unique()
)

# Tập Item có Category
category_items = set(
    item_category_history_final["itemid"].unique()
)

total_event_items = len(event_items)

# =========================================================
# 1. Item Event có Property
# =========================================================

event_items_with_property = (
    len(event_items & property_items)
)

event_items_without_property = (
    len(event_items - property_items)
)

print("\n1. Item Event có Property:")
print(event_items_with_property)

print("Item Event không có Property:")
print(event_items_without_property)

print("Tỷ lệ Item Event có Property:")
print(
    round(
        event_items_with_property
        / total_event_items * 100,
        2
    ),
    "%"
)

# =========================================================
# 2. Item Event có Category
# =========================================================

event_items_with_category = (
    len(event_items & category_items)
)

event_items_without_category = (
    len(event_items - category_items)
)

print("\n2. Item Event có Category:")
print(event_items_with_category)

print("Item Event không có Category:")
print(event_items_without_category)

print("Tỷ lệ Item Event có Category:")
print(
    round(
        event_items_with_category
        / total_event_items * 100,
        2
    ),
    "%"
)

# =========================================================
# 3. Event coverage của Properties
# =========================================================

events_with_property_item = (
    user_item_event_final["itemid"]
    .isin(property_items)
)

print("\n3. Event thuộc Item có Property:")
print(
    events_with_property_item.sum()
)

print("Tỷ lệ Event thuộc Item có Property:")
print(
    round(
        events_with_property_item.mean() * 100,
        2
    ),
    "%"
)

# =========================================================
# 4. Event coverage của Category
# =========================================================

events_with_category_item = (
    user_item_event_final["itemid"]
    .isin(category_items)
)

print("\n4. Event thuộc Item có Category:")
print(
    events_with_category_item.sum()
)

print("Tỷ lệ Event thuộc Item có Category:")
print(
    round(
        events_with_category_item.mean() * 100,
        2
    ),
    "%"
)

# =========================================================
# 5. Số Property trên mỗi Item
# =========================================================

properties_per_item = (
    item_properties_final
    .groupby("itemid")["property"]
    .nunique()
)

print("\n5. Số Property khác nhau trên mỗi Item:")
print(
    properties_per_item.describe()
)

# =========================================================
# 6. Số Category trên mỗi Item
# =========================================================

categories_per_item = (
    item_category_history_final
    .groupby("itemid")["categoryid"]
    .nunique()
)

print("\n6. Số Category khác nhau trên mỗi Item:")
print(
    categories_per_item.describe()
)

# =========================================================
# 7. Item có nhiều Category
# =========================================================

items_multiple_categories = (
    (categories_per_item > 1)
    .sum()
)

print("\n7. Item có nhiều hơn 1 Category:")
print(items_multiple_categories)

print("Tỷ lệ:")
print(
    round(
        items_multiple_categories
        / len(categories_per_item) * 100,
        2
    ),
    "%"
)

# =========================================================
# 8. Category không có trong Category Tree
# =========================================================

category_tree_ids = set(
    category_tree_final["categoryid"]
)

property_category_ids = set(
    item_category_history_final["categoryid"]
)

categories_without_tree = (
    property_category_ids - category_tree_ids
)

print("\n8. Category có trong Item Properties nhưng không có trong Tree:")
print(len(categories_without_tree))

print(
    sorted(categories_without_tree)
)

# =========================================================
# 9. Kiểm tra Item Properties theo thời gian
# =========================================================

print("\n9. Khoảng thời gian Item Properties:")
print(
    item_properties_final["datetime"].min(),
    "→",
    item_properties_final["datetime"].max()
)

print("\nKhoảng thời gian Event:")
print(
    user_item_event_final["datetime"].min(),
    "→",
    user_item_event_final["datetime"].max()
)